In [ ]:
from datasets import load_dataset

ds = load_dataset("sewon/ambig_qa", "light")

ModuleNotFoundError: No module named 'datasets'

In [ ]:
data = ds['train']
print(len(data))

In [ ]:
data[1]

NameError: name 'data' is not defined

In [ ]:
import pandas as pd
import json

# --- 1. Load your data (if not already loaded) ---
# data = [json.loads(line) for line in open("ambignq_train_data.json")]

rows = []
for entry in data:
    ann = entry["annotations"]

    # Extract question type (string)
    ann_type = ann["type"][0] if isinstance(ann["type"], list) else ann["type"]

    # Extract qaPairs safely
    qa_pairs = ann.get("qaPairs", [])
    if len(qa_pairs) == 1 and isinstance(qa_pairs[0], list):
        qa_pairs = qa_pairs[0]

    # Extract single-answer list (only for singleAnswer entries)
    single_ans_list = ann.get("answer", [])
    single_ans_flat = []
    if isinstance(single_ans_list, list):
        for ans in single_ans_list:
            if isinstance(ans, list):
                single_ans_flat.extend(ans)
            elif isinstance(ans, str):
                single_ans_flat.append(ans)

    row = {
        "question": entry["question"],
        "type": ann_type,
        "qaPairs": qa_pairs,
        "single_answers": single_ans_flat if ann_type == "singleAnswer" else []
    }
    rows.append(row)

og_df = pd.DataFrame(rows)

# =========================================================
# 🔹 Extract Disambiguated Questions
# =========================================================
def extract_disambig_questions(qa_pairs):
    disambig = []
    if not isinstance(qa_pairs, list):
        return disambig
    for pair in qa_pairs:
        if not isinstance(pair, dict):
            continue
        qs = pair.get("question", [])
        if isinstance(qs, list):
            disambig.extend(qs)
        elif isinstance(qs, str):
            disambig.append(qs)
    return disambig


# =========================================================
# 🔹 Extract Disambiguated Answers
# =========================================================
def extract_disambig_answers(qa_pairs):
    answers = []
    if not isinstance(qa_pairs, list):
        return answers
    for pair in qa_pairs:
        if not isinstance(pair, dict):
            continue
        ans_list = pair.get("answer", [])
        if isinstance(ans_list, list):
            for a in ans_list:
                if isinstance(a, list):
                    answers.extend(a)
                elif isinstance(a, str):
                    answers.append(a)
    return answers


# =========================================================
# 🔹 Apply extraction to DataFrame
# =========================================================
og_df["disambig_questions"] = og_df["qaPairs"].apply(extract_disambig_questions)
og_df["disambig_answers"] = og_df["qaPairs"].apply(extract_disambig_answers)
og_df["is_ambiguous"] = og_df["type"].apply(lambda t: t == "multipleQAs")

# --- For singleAnswer entries, copy their answers from annotations ---
og_df["single_answers"] = og_df.apply(
    lambda r: r["single_answers"] if r["type"] == "singleAnswer" else [], axis=1
)

# =========================================================
# ✅ Sanity Check
# =========================================================
print(og_df[["question", "type", "disambig_questions", "disambig_answers", "single_answers", "is_ambiguous"]].head(5))


                                            question          type  \
0     When did the simpsons first air on television?   multipleQAs   
1  Who played george washington in the john adams...  singleAnswer   
2          What is the legal age of marriage in usa?   multipleQAs   
3   Who starred in barefoot in the park on broadway?   multipleQAs   
4      When did the manhattan project began and end?   multipleQAs   

                                  disambig_questions  \
0  [When did the Simpsons first air on television...   
1                                                 []   
2  [What is the legal age of marriage, without pa...   
3  [Who starred in barefoot in the park on broadw...   
4  [Based on the initial thoughts of the project,...   

                                    disambig_answers single_answers  \
0                [April 19, 1987, December 17, 1989]             []   
1                                                 []  [David Morse]   
2                   [18 years

In [ ]:
og_df

,question,type,qaPairs,single_answers,disambig_questions,disambig_answers,is_ambiguous
0,When did the simpsons first air on television?,multipleQAs,[{'question': ['When did the Simpsons first ai...,[],[When did the Simpsons first air on television...,"[April 19, 1987, December 17, 1989]",True
1,Who played george washington in the john adams...,singleAnswer,"[{'question': [], 'answer': []}]",[David Morse],[],[],False
2,What is the legal age of marriage in usa?,multipleQAs,[{'question': ['What is the legal age of marri...,[],"[What is the legal age of marriage, without pa...","[18 years of age, 18, 19, 21, 0]",True
3,Who starred in barefoot in the park on broadway?,multipleQAs,[{'question': ['Who starred in barefoot in the...,[],[Who starred in barefoot in the park on broadw...,"[Elizabeth Ashley, Kurt Kasznar, Mildred Natwi...",True
4,When did the manhattan project began and end?,multipleQAs,[{'question': ['Based on the initial thoughts ...,[],"[Based on the initial thoughts of the project,...","[Began 1939, end 1946, Began 1942, end 1946]",True
...,...,...,...,...,...,...,...
10031,When do the summer holidays start for schools?,multipleQAs,[{'question': ['When do summer holidays start ...,[],[When do summer holidays start for schools in ...,"[end of June, between late-May and mid-June, B...",True
10032,Who is the band in the movie 10 things i hate ...,multipleQAs,[{'question': ['What band plays at Club Skunk ...,[],[What band plays at Club Skunk in the movie 10...,"[Letters to Cleo, Save Ferris, Letters to Cleo...",True
10033,Who was the last person in the uk to be executed?,singleAnswer,"[{'question': [], 'answer': []}]",[Gwynne Evans and Peter Allen],[],[],False
10034,Who does wonder woman end up with in the comics?,multipleQAs,[{'question': ['Who does wonder woman end up w...,[],[Who does wonder woman end up with in All Star...,"[General Steven Rockwell Trevor, Steve Trevor,...",True


Positive answers for disambiguous questions

In [ ]:
full_facet_df=pd.read_csv('full_reasoning_df.csv')
full_facet_df.head()

,Unnamed: 0,question,type,qaPairs,disambig_questions,is_ambiguous,facets,reasoning
0,0,When did the simpsons first air on television?,multipleQAs,[{'question': ['When did the Simpsons first ai...,['When did the Simpsons first air on televisio...,True,"['Type of Show', 'Platform']",The question is ambiguous because it does not ...
1,1,Who played george washington in the john adams...,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],The question is not ambiguous because it speci...
2,2,What is the legal age of marriage in usa?,multipleQAs,[{'question': ['What is the legal age of marri...,"['What is the legal age of marriage, without p...",True,"['State', 'Circumstances for Marriage']",The question is ambiguous because the legal ag...
3,3,Who starred in barefoot in the park on broadway?,multipleQAs,[{'question': ['Who starred in barefoot in the...,['Who starred in barefoot in the park on broad...,True,['Character Name'],The question is ambiguous because it does not ...
4,4,When did the manhattan project began and end?,multipleQAs,[{'question': ['Based on the initial thoughts ...,['Based on the initial thoughts of the project...,True,['Time Frame Interpretation'],The question is ambiguous because it does not ...


In [ ]:
og_df

,question,type,qaPairs,single_answers,disambig_questions,disambig_answers,is_ambiguous
0,When did the simpsons first air on television?,multipleQAs,[{'question': ['When did the Simpsons first ai...,[],[When did the Simpsons first air on television...,"[April 19, 1987, December 17, 1989]",True
1,Who played george washington in the john adams...,singleAnswer,"[{'question': [], 'answer': []}]",[David Morse],[],[],False
2,What is the legal age of marriage in usa?,multipleQAs,[{'question': ['What is the legal age of marri...,[],"[What is the legal age of marriage, without pa...","[18 years of age, 18, 19, 21, 0]",True
3,Who starred in barefoot in the park on broadway?,multipleQAs,[{'question': ['Who starred in barefoot in the...,[],[Who starred in barefoot in the park on broadw...,"[Elizabeth Ashley, Kurt Kasznar, Mildred Natwi...",True
4,When did the manhattan project began and end?,multipleQAs,[{'question': ['Based on the initial thoughts ...,[],"[Based on the initial thoughts of the project,...","[Began 1939, end 1946, Began 1942, end 1946]",True
...,...,...,...,...,...,...,...
10031,When do the summer holidays start for schools?,multipleQAs,[{'question': ['When do summer holidays start ...,[],[When do summer holidays start for schools in ...,"[end of June, between late-May and mid-June, B...",True
10032,Who is the band in the movie 10 things i hate ...,multipleQAs,[{'question': ['What band plays at Club Skunk ...,[],[What band plays at Club Skunk in the movie 10...,"[Letters to Cleo, Save Ferris, Letters to Cleo...",True
10033,Who was the last person in the uk to be executed?,singleAnswer,"[{'question': [], 'answer': []}]",[Gwynne Evans and Peter Allen],[],[],False
10034,Who does wonder woman end up with in the comics?,multipleQAs,[{'question': ['Who does wonder woman end up w...,[],[Who does wonder woman end up with in All Star...,"[General Steven Rockwell Trevor, Steve Trevor,...",True


In [ ]:
pos_ans=full_facet_df.merge(og_df[['question','single_answers']], how='inner', on='question')
pos_ans

,Unnamed: 0,question,type,qaPairs,disambig_questions,is_ambiguous,facets,reasoning,single_answers
0,0,When did the simpsons first air on television?,multipleQAs,[{'question': ['When did the Simpsons first ai...,['When did the Simpsons first air on televisio...,True,"['Type of Show', 'Platform']",The question is ambiguous because it does not ...,[]
1,1,Who played george washington in the john adams...,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],The question is not ambiguous because it speci...,[David Morse]
2,2,What is the legal age of marriage in usa?,multipleQAs,[{'question': ['What is the legal age of marri...,"['What is the legal age of marriage, without p...",True,"['State', 'Circumstances for Marriage']",The question is ambiguous because the legal ag...,[]
3,3,Who starred in barefoot in the park on broadway?,multipleQAs,[{'question': ['Who starred in barefoot in the...,['Who starred in barefoot in the park on broad...,True,['Character Name'],The question is ambiguous because it does not ...,[]
4,4,When did the manhattan project began and end?,multipleQAs,[{'question': ['Based on the initial thoughts ...,['Based on the initial thoughts of the project...,True,['Time Frame Interpretation'],The question is ambiguous because it does not ...,[]
...,...,...,...,...,...,...,...,...,...
10031,10031,When do the summer holidays start for schools?,multipleQAs,[{'question': ['When do summer holidays start ...,['When do summer holidays start for schools in...,True,"['Geographical Region', 'Specific Location']",The question is ambiguous because it does not ...,[]
10032,10032,Who is the band in the movie 10 things i hate ...,multipleQAs,[{'question': ['What band plays at Club Skunk ...,['What band plays at Club Skunk in the movie 1...,True,"['Location', 'Event Type']",The question is ambiguous because it does not ...,[]
10033,10033,Who was the last person in the uk to be executed?,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],The question is not ambiguous because it speci...,[Gwynne Evans and Peter Allen]
10034,10034,Who does wonder woman end up with in the comics?,multipleQAs,[{'question': ['Who does wonder woman end up w...,['Who does wonder woman end up with in All Sta...,True,"['Story Universe', 'Publication Series']",The question is ambiguous because 'Wonder Woma...,[]


Generating positive responses for ambiguous questions

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# # Choose model
# model_name = "Qwen/Qwen2.5-7B-Instruct"

# # Load tokenizer and model
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.bfloat16)

# # Move to GPU if available
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model = model.to(device)

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",          # let Accelerate place layers
    torch_dtype=torch.bfloat16, # good precision for A100/T4
)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
df=pos_ans.copy()
df=df[:990]
print(df.shape)
df.head()


(990, 9)


,Unnamed: 0,question,type,qaPairs,disambig_questions,is_ambiguous,facets,reasoning,single_answers
0,0,When did the simpsons first air on television?,multipleQAs,[{'question': ['When did the Simpsons first ai...,['When did the Simpsons first air on televisio...,True,"['Type of Show', 'Platform']",The question is ambiguous because it does not ...,[]
1,1,Who played george washington in the john adams...,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],The question is not ambiguous because it speci...,[David Morse]
2,2,What is the legal age of marriage in usa?,multipleQAs,[{'question': ['What is the legal age of marri...,"['What is the legal age of marriage, without p...",True,"['State', 'Circumstances for Marriage']",The question is ambiguous because the legal ag...,[]
3,3,Who starred in barefoot in the park on broadway?,multipleQAs,[{'question': ['Who starred in barefoot in the...,['Who starred in barefoot in the park on broad...,True,['Character Name'],The question is ambiguous because it does not ...,[]
4,4,When did the manhattan project began and end?,multipleQAs,[{'question': ['Based on the initial thoughts ...,['Based on the initial thoughts of the project...,True,['Time Frame Interpretation'],The question is ambiguous because it does not ...,[]


In [ ]:
# ==============================================================
# 🔹 IMPORTS & SETUP
# ==============================================================
import re
import ast
import gc
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---- Device Setup ----
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


# ==============================================================
# 🔹 Helper: Extract only [OUTPUT] content
# ==============================================================
def extract_output(text):
    """
    Extracts the model-generated clarifying question from between [OUTPUT] tags.
    """
    m = re.search(r"\[OUTPUT\](.*?)\[/OUTPUT\]", text, flags=re.DOTALL)
    if m:
        return m.group(1).strip()
    else:
        # fallback — return last few lines if no tags
        lines = text.strip().splitlines()
        return lines[-1].strip() if lines else text.strip()

# ==============================================================
# 🔹 Build Positive Clarifying Prompt
# ==============================================================
def build_positive_prompt(question, facets):
    facets_txt = ", ".join(facets) if facets else "N/A"
    #tax_txt = ", ".join(taxonomies) if taxonomies else "N/A"

    return f"""
You are a helpful assistant that clarifies ambiguous questions.

Your task: Ask a **single-turn clarifying question** that helps the user
specify their intent and resolve all the ambiguity. Use the provided facets (as clues about *what* is ambiguous)

---

**Ambiguous Question:**
{question}

**Facets (dimensions of ambiguity):**
{facets_txt}

---

Guidelines:
1. Ask one polite, natural question that helps clarify **all listed facets**.
2. Do **not** answer the question.
3. Keep your response short (1 sentence max).
4. Output the clarifying question only between [OUTPUT] and [/OUTPUT].

---

### Example 1
Ambiguous: "When did The Simpsons first air?"
Facets: Type of Show, Broadcast Region
Output: [OUTPUT]Could you clarify which type of Simpsons show you’re referring to, and in which region it aired?[/OUTPUT]

### Example 2
Ambiguous: "Who won the US Open?"
Facets: Sport Type, Year
Output: [OUTPUT]Do you mean the US Open in tennis or golf, and which year are you asking about?[/OUTPUT]

---

Now write your clarifying question.
Remember to wrap it exactly as [OUTPUT] ... [/OUTPUT].
"""

# ==============================================================
# 🔹 LLM Call Utility
# ==============================================================
def generate_text(prompt, temperature=0.5, max_new_tokens=100, debug=False):
    """
    Generates only the model's clarifying question, not the prompt.
    """
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3000).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True
        )

    # Decode only the newly generated part, not the input prompt
    gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(gen_tokens, skip_special_tokens=True)

    if debug:
        print("\n=== RAW MODEL OUTPUT ===\n", text, "\n=========================\n")

    return extract_output(text)

# ==============================================================
# 🔹 MAIN LOOP — GENERATE POSITIVE RESPONSES ONLY
# ==============================================================
positive_responses = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Generating positive clarifying questions"):
    if not row.get("is_ambiguous", False):
        positive_responses.append("N/A")
        continue

    question = row["question"]

    # --- Parse facet_taxonomies safely ---
    facets_raw = row.get("facets", [])
    if isinstance(facets_raw, str):
        try:
            facets = ast.literal_eval(facets_raw)
        except Exception:
            facets = [facets_raw]
    elif isinstance(facets_raw, list):
        facets = facets_raw
    else:
        facets = []

    #taxonomies = [ft.get("taxonomy", "") for ft in raw_ft if isinstance(ft, dict)]

    try:
        pos_prompt = build_positive_prompt(question, facets)
        pos_out = generate_text(pos_prompt, temperature=0.6, debug=True)
        positive_responses.append(pos_out)
    except torch.cuda.OutOfMemoryError:
        print(f"\n⚠️ OOM at row {idx}, switching to CPU temporarily...")
        model.to("cpu")
        pos_out = generate_text(pos_prompt, temperature=0.6, debug=False)
        positive_responses.append(pos_out)
        model.to(device)
    finally:
        torch.cuda.empty_cache()
        gc.collect()

    # --- Print summary per question ---
    print(f"\n🟩 Row {idx}")
    print(f"Q: {question}")
    print(f"Facets: {facets}")
    #print(f"Taxonomies: {taxonomies}")
    print(f"✅ Clarifying Question: {pos_out}")
    print("------------------------------------------------------\n")

# ==============================================================
# 🔹 SAVE OUTPUT
# ==============================================================
df["positive_response"] = positive_responses
df.to_csv("positive_clarifying_responses.csv", index=False)
print("\n✅ Saved results to positive_clarifying_responses.csv")


Using device: cuda


Generating positive clarifying questions:   0%|          | 0/990 [00:00<?, ?it/s]


=== RAW MODEL OUTPUT ===
 [OUTPUT]Are you asking about the original TV series or a movie, and on which platform (e.g., network, streaming service) did it first air?[/OUTPUT] 



Generating positive clarifying questions:   0%|          | 1/990 [00:01<26:50,  1.63s/it]


🟩 Row 0
Q: When did the simpsons first air on television?
Facets: ['Type of Show', 'Platform']
✅ Clarifying Question: Are you asking about the original TV series or a movie, and on which platform (e.g., network, streaming service) did it first air?
------------------------------------------------------


=== RAW MODEL OUTPUT ===
 [OUTPUT]In which U.S. state and under what circumstances are you asking about the legal age of marriage?[/OUTPUT] 


🟩 Row 2
Q: What is the legal age of marriage in usa?
Facets: ['State', 'Circumstances for Marriage']
✅ Clarifying Question: In which U.S. state and under what circumstances are you asking about the legal age of marriage?
------------------------------------------------------



...


🟩 Row 979
Q: Where does rome's drinking water come from?
Facets: ['Time Period', 'Type of Water Source', 'Transportation Method']
✅ Clarifying Question: Are you asking about the historical sources of Rome's drinking water, the current sources, or both; what kind of water source (e.g., aqueducts, wells) are you interested in; and how do you want the information presented (e.g., through ancient records, modern methods)?
------------------------------------------------------



Generating positive clarifying questions:  99%|█████████▉| 984/990 [09:09<00:03,  1.77it/s]


=== RAW MODEL OUTPUT ===
 [OUTPUT]Are you asking about the type of alpha amylase, and in which specific part of the body it is found?[/OUTPUT] 


🟩 Row 983
Q: Where is alpha amylase found in the body?
Facets: ['Type of Alpha Amylase', 'Location in the Body']
✅ Clarifying Question: Are you asking about the type of alpha amylase, and in which specific part of the body it is found?
------------------------------------------------------



Generating positive clarifying questions: 100%|██████████| 990/990 [09:10<00:00,  1.80it/s]


=== RAW MODEL OUTPUT ===
 [OUTPUT]Are you asking about the total number of NBA Defensive Player of the Year awards won by an individual player, considering both consistency and continuity of the award over time?[/OUTPUT] 


🟩 Row 987
Q: Who has the most nba defensive player of the year awards?
Facets: ['Team Consistency', 'Award Continuity']
✅ Clarifying Question: Are you asking about the total number of NBA Defensive Player of the Year awards won by an individual player, considering both consistency and continuity of the award over time?
------------------------------------------------------


✅ Saved results to positive_clarifying_responses.csv


In [ ]:
df

,Unnamed: 0,question,type,qaPairs,disambig_questions,is_ambiguous,facets,reasoning,single_answers,positive_response
990,990,Kansas city chief player who is a doctor?,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],The question is not ambiguous because it speci...,"[Laurent Duvernay-Tardif CQ, Laurent Duvernay-...",N/A
991,991,When did the last civil war widow die?,multipleQAs,[{'question': ['When did the last civil war wi...,['When did the last civil war widow from the U...,True,['Side of Civil War'],The question is ambiguous because it does not ...,[],Could you specify which side of the Civil War ...
992,992,What does sho stand for on ford taurus?,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],The question is not ambiguous because it speci...,[Super High Output],N/A
993,993,Who won australia's next top model season 7?,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],The question is not ambiguous because it speci...,"[Montana Cox, Cox]",N/A
994,994,What is the climate in the gobi desert?,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],The question is not ambiguous because it speci...,"[a rain shadow desert, great extremes, combine...",N/A
...,...,...,...,...,...,...,...,...,...,...
10031,10031,When do the summer holidays start for schools?,multipleQAs,[{'question': ['When do summer holidays start ...,['When do summer holidays start for schools in...,True,"['Geographical Region', 'Specific Location']",The question is ambiguous because it does not ...,[],In which geographical region and specific loca...
10032,10032,Who is the band in the movie 10 things i hate ...,multipleQAs,[{'question': ['What band plays at Club Skunk ...,['What band plays at Club Skunk in the movie 1...,True,"['Location', 'Event Type']",The question is ambiguous because it does not ...,[],Are you asking about the band featured in the ...
10033,10033,Who was the last person in the uk to be executed?,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],The question is not ambiguous because it speci...,[Gwynne Evans and Peter Allen],N/A
10034,10034,Who does wonder woman end up with in the comics?,multipleQAs,[{'question': ['Who does wonder woman end up w...,['Who does wonder woman end up with in All Sta...,True,"['Story Universe', 'Publication Series']",The question is ambiguous because 'Wonder Woma...,[],Are you asking about Wonder Woman's relationsh...


## concat the first 1k df with the rest 9k df

In [ ]:
import pandas as pd
import numpy as np

first_half=pd.read_csv('positive_clarifying_responses-4.csv')
second_half=pd.read_csv('positive_clarifying_responses-3.csv')

print(first_half.shape)
print(second_half.shape)

(990, 10)
(9046, 10)


In [ ]:
first_half.tail()

,Unnamed: 0,question,type,qaPairs,disambig_questions,is_ambiguous,facets,reasoning,single_answers,positive_response
985,985,How many pages in a brave new world?,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],The question is not ambiguous because it speci...,['311'],NaN
986,986,When was the last time newcastle won the fa cup?,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],The question is not ambiguous because it speci...,['1954-55'],NaN
987,987,Who has the most nba defensive player of the y...,multipleQAs,[{'question': ['Who has the most nba defensive...,['Who has the most nba defensive player of the...,True,"['Team Consistency', 'Award Continuity']",The question is ambiguous because it does not ...,[],Are you asking about the total number of NBA D...
988,988,What are the components of the renal corpuscle?,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],The question is not ambiguous because it speci...,"[""glomerulus, Bowman's capsule""]",NaN
989,989,Who sang bird bird bird bird is the word?,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],The question is not ambiguous because it provi...,['The Trashmen'],NaN


In [ ]:
second_half.tail()

,Unnamed: 0,question,type,qaPairs,disambig_questions,is_ambiguous,facets,reasoning,single_answers,positive_response
9041,10031,When do the summer holidays start for schools?,multipleQAs,[{'question': ['When do summer holidays start ...,['When do summer holidays start for schools in...,True,"['Geographical Region', 'Specific Location']",The question is ambiguous because it does not ...,[],In which geographical region and specific loca...
9042,10032,Who is the band in the movie 10 things i hate ...,multipleQAs,[{'question': ['What band plays at Club Skunk ...,['What band plays at Club Skunk in the movie 1...,True,"['Location', 'Event Type']",The question is ambiguous because it does not ...,[],Are you asking about the band featured in the ...
9043,10033,Who was the last person in the uk to be executed?,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],The question is not ambiguous because it speci...,['Gwynne Evans and Peter Allen'],NaN
9044,10034,Who does wonder woman end up with in the comics?,multipleQAs,[{'question': ['Who does wonder woman end up w...,['Who does wonder woman end up with in All Sta...,True,"['Story Universe', 'Publication Series']",The question is ambiguous because 'Wonder Woma...,[],Are you asking about Wonder Woman's relationsh...
9045,10035,When were the first pair of jordans released?,multipleQAs,[{'question': ['When were the first pair of Ai...,['When were the first pair of Air Jordans rele...,True,"['Release Target Audience', 'Release Scope']",The question is ambiguous because it does not ...,[],Could you specify whether you're asking about ...


In [ ]:
final_df = pd.concat([first_half, second_half], ignore_index=True)

# Step 2: Drop duplicates (keeps the first occurrence)
final_df = final_df.drop_duplicates(ignore_index=True)

# Check final shape
print(final_df.shape)

final_df.head()

(10036, 10)


,Unnamed: 0,question,type,qaPairs,disambig_questions,is_ambiguous,facets,reasoning,single_answers,positive_response
0,0,When did the simpsons first air on television?,multipleQAs,[{'question': ['When did the Simpsons first ai...,['When did the Simpsons first air on televisio...,True,"['Type of Show', 'Platform']",The question is ambiguous because it does not ...,[],Are you asking about the original TV series or...
1,1,Who played george washington in the john adams...,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],The question is not ambiguous because it speci...,['David Morse'],NaN
2,2,What is the legal age of marriage in usa?,multipleQAs,[{'question': ['What is the legal age of marri...,"['What is the legal age of marriage, without p...",True,"['State', 'Circumstances for Marriage']",The question is ambiguous because the legal ag...,[],In which U.S. state and under what circumstanc...
3,3,Who starred in barefoot in the park on broadway?,multipleQAs,[{'question': ['Who starred in barefoot in the...,['Who starred in barefoot in the park on broad...,True,['Character Name'],The question is ambiguous because it does not ...,[],Did you mean the character names or the actors...
4,4,When did the manhattan project began and end?,multipleQAs,[{'question': ['Based on the initial thoughts ...,['Based on the initial thoughts of the project...,True,['Time Frame Interpretation'],The question is ambiguous because it does not ...,[],In which specific year or time period are you ...


In [ ]:
final_df['action']=np.where(final_df['is_ambiguous']==True, 'Clarify', 'Answer')

In [ ]:
final_df.to_csv('final_SFT_training_data.csv')

## SETTING UP DATA FOR TRAINING PHASE

In [ ]:
final_df.columns

Index(['Unnamed: 0', 'question', 'type', 'qaPairs', 'disambig_questions',
       'is_ambiguous', 'facets', 'reasoning', 'single_answers',
       'positive_response', 'action'],
      dtype='object')

In [ ]:
final_df['positive_response']=final_df['positive_response'].fillna(final_df['single_answers'])

In [ ]:
final_sft_df=final_df[['question', 'is_ambiguous', 'facets', 'reasoning', 'positive_response', 'action']]
pd.set_option('display.max_colwidth', None)
final_sft_df.head()

,question,is_ambiguous,facets,reasoning,positive_response,action
0,When did the simpsons first air on television?,True,"['Type of Show', 'Platform']","The question is ambiguous because it does not specify whether 'The Simpsons' refers to the TV series or another type of show, and it does not clarify if the platform (e.g., network, streaming service) is relevant. Without this information, we cannot determine the exact date of its first broadcast.","Are you asking about the original TV series or a movie, and on which platform (e.g., network, streaming service) did it first air?",Clarify
1,Who played george washington in the john adams series?,False,[],"The question is not ambiguous because it specifies the exact role and series, providing enough detail to uniquely identify the actor who portrayed George Washington in the John Adams miniseries.",['David Morse'],Answer
2,What is the legal age of marriage in usa?,True,"['State', 'Circumstances for Marriage']","The question is ambiguous because the legal age of marriage can vary by state in the USA. Additionally, there might be different circumstances or exceptions that affect the age requirement, such as parental consent or a court order. Without specifying the state and the circumstances, the answer can differ significantly.",In which U.S. state and under what circumstances are you asking about the legal age of marriage?,Clarify
3,Who starred in barefoot in the park on broadway?,True,['Character Name'],"The question is ambiguous because it does not specify which character from the play ""Barefoot in the Park"" was starred in. Without knowing the character name, we cannot determine who played that specific role on Broadway.",Did you mean the character names or the actors who played them in the Broadway production of Barefoot in the Park?,Clarify
4,When did the manhattan project began and end?,True,['Time Frame Interpretation'],"The question is ambiguous because it does not specify which aspect of the Manhattan Project is being referred to—whether it's the start date, end date, or a broader time frame such as the duration. The term ""began and ended"" could imply either two separate dates or the entire period of the project.",In which specific year or time period are you interested in knowing about the Manhattan Project's start and end?,Clarify


In [ ]:
final_sft_df.to_csv('final_SFT_data_for_use.csv')

In [ ]:
import pandas as pd
final_sft_df=pd.read_csv('final_SFT_data_for_use.csv')

In [ ]:
#results_primitive.to_csv('pos_amb_disambig_answers.csv')

### Pre-process the data to get it into proper json format

In [ ]:
# import json
# import ast
# import pandas as pd

# df = final_sft_df.copy()

# def clean_facets(facets):
#     if isinstance(facets, list):
#         return facets
#     if isinstance(facets, str):
#         try:
#             val = ast.literal_eval(facets)
#             return val if isinstance(val, list) else []
#         except:
#             return []
#     return []

# def clean_response(resp):
#     if resp is None:
#         return ""

#     # If it's a list, return the first element
#     if isinstance(resp, list):
#         return resp[0] if len(resp) > 0 else ""

#     # If it's a string of a list, extract content
#     if isinstance(resp, str) and resp.startswith("[") and resp.endswith("]"):
#         try:
#             list_val = ast.literal_eval(resp)
#             if isinstance(list_val, list) and len(list_val) > 0:
#                 return list_val[0]
#         except:
#             pass

#     # Otherwise return clean string
#     return resp.strip()


# records = []

# for _, row in df.iterrows():
#     q = row["question"]
#     is_amb = row["is_ambiguous"]
#     reasoning = row["reasoning"]
#     action = row["action"].strip()
#     facets = clean_facets(row["facets"])

#     # Clarify vs Answer
#     response_raw = row["positive_response"]
#     response = clean_response(response_raw)

#     record = {
#         "instruction": (
#             "Decide whether the user's question is ambiguous. "
#             "If ambiguous, explain the ambiguity, list facets, and generate a clarifying question. "
#             "If not ambiguous, explain why and answer directly."
#         ),
#         "input": q,
#         "output": {
#             "action": action,
#             "reasoning": reasoning,
#             "facets": facets if is_amb else [],
#             "response": response
#         }
#     }
#     records.append(record)

# with open("sft_data.jsonl", "w") as f:
#     for r in records:
#         f.write(json.dumps(r) + "\n")

# print("⭐ CLEAN SFT dataset created: sft_data.jsonl")


In [ ]:
# import json

# with open("sft_data.jsonl") as f:
#     for _ in range(5):
#         print(json.loads(next(f)))
#         print("------")


In [ ]:
!pip install torch transformers>=4.37.0 datasets peft trl accelerate bitsandbytes sentencepiece protobuf einops

In [ ]:
# # ============================================================================
# # COMPLETE QWEN SFT TRAINING PIPELINE (FINAL STABLE VERSION)
# # ============================================================================

# import json
# import ast
# import pandas as pd
# import torch
# from datasets import load_dataset
# from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
# from peft import LoraConfig
# from trl import SFTTrainer

# print("\n" + "="*80)
# print("STEP 1: PREPARING DATASET")
# print("="*80)

# # Use your dataframe
# df = final_sft_df.copy()

# # ----------------------------------------------------------
# # Cleaning helpers
# # ----------------------------------------------------------

# def clean_facets(facets):
#     if isinstance(facets, list):
#         return facets
#     if isinstance(facets, str):
#         try:
#             val = ast.literal_eval(facets)
#             return val if isinstance(val, list) else []
#         except:
#             return []
#     return []

# def clean_response(resp):
#     if resp is None:
#         return ""
#     if isinstance(resp, list):
#         return resp[0] if resp else ""
#     if isinstance(resp, str) and resp.startswith("[") and resp.endswith("]"):
#         try:
#             lst = ast.literal_eval(resp)
#             return lst[0] if lst else ""
#         except:
#             pass
#     return str(resp).strip()

# # ----------------------------------------------------------
# # Build SFT JSONL
# # ----------------------------------------------------------

# records = []

# for _, row in df.iterrows():
#     q = row["question"]
#     is_amb = row["is_ambiguous"]
#     reasoning = row["reasoning"]
#     action = row["action"]
#     facets = clean_facets(row["facets"])
#     response = clean_response(row["positive_response"])

#     records.append({
#         "instruction": (
#             "Decide whether the user's question is ambiguous. "
#             "If ambiguous, explain the ambiguity, list facets, and ask a clarifying question. "
#             "If not ambiguous, explain why and answer directly."
#         ),
#         "input": q,
#         "output": {
#             "action": action,
#             "reasoning": reasoning,
#             "facets": facets if is_amb else [],
#             "response": response
#         }
#     })

# with open("sft_data.jsonl", "w") as f:
#     for r in records:
#         f.write(json.dumps(r) + "\n")

# print(f"✅ Created SFT dataset with {len(records)} rows")


# # ============================================================================
# # STEP 2: LOAD DATASET
# # ============================================================================

# print("\n" + "="*80)
# print("STEP 2: LOADING DATASET")
# print("="*80)

# train_dataset = load_dataset("json", data_files="sft_data.jsonl")["train"]
# print(f"Loaded {len(train_dataset)} examples")


# # ============================================================================
# # STEP 3: LOAD MODEL + TOKENIZER
# # ============================================================================

# print("\n" + "="*80)
# print("STEP 3: LOADING MODEL & TOKENIZER")
# print("="*80)

# model_name = "Qwen/Qwen2.5-7B-Instruct"

# tokenizer = AutoTokenizer.from_pretrained(
#     model_name,
#     trust_remote_code=True
# )

# # Critical fix
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "right"

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
#     trust_remote_code=True
# )

# model.config.use_cache = False
# model.gradient_checkpointing_enable()


# # ============================================================================
# # STEP 4: FORMAT TEXT FOR TRAINING USING CHAT TEMPLATE
# # ============================================================================

# print("\n" + "="*80)
# print("STEP 4: FORMATTING DATASET (CHAT TEMPLATE)")
# print("="*80)

# def format_chat(example):
#     question = example["input"]
#     action = example["output"]["action"]
#     reasoning = example["output"]["reasoning"]
#     facets = example["output"]["facets"]
#     response = example["output"]["response"]

#     facets_str = "[" + ", ".join(facets) + "]" if facets else "[]"

#     system_msg = (
#         "You are a question understanding agent. For each user question:\n"
#         "1) Decide if it is ambiguous.\n"
#         "2) Explain your reasoning.\n"
#         "3) List facets if ambiguous.\n"
#         "4) Ask a clarifying question OR give a direct answer.\n\n"
#         "Format:\n"
#         "Action: Clarify|Answer\n"
#         "Reasoning: ...\n"
#         "Facets: [...]\n"
#         "Response: ...\n"
#     )

#     assistant_msg = (
#         f"Action: {action}\n"
#         f"Reasoning: {reasoning}\n"
#         f"Facets: {facets_str}\n"
#         f"Response: {response}"
#     )

#     messages = [
#         {"role": "system", "content": system_msg},
#         {"role": "user", "content": question},
#         {"role": "assistant", "content": assistant_msg}
#     ]

#     formatted = tokenizer.apply_chat_template(
#         messages, tokenize=False, add_generation_prompt=False
#     )
#     return {"text": formatted}

# formatted_dataset = train_dataset.map(
#     format_chat,
#     remove_columns=train_dataset.column_names
# )

# print("Example formatted text:")
# print("-"*80)
# print(formatted_dataset[0]["text"][:400])
# print("-"*80)


# # ============================================================================
# # STEP 5: LoRA CONFIG
# # ============================================================================

# print("\n" + "="*80)
# print("STEP 5: LORA CONFIG")
# print("="*80)

# lora_config = LoraConfig(
#     r=32,
#     lora_alpha=16,
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM",
#     target_modules=[
#         "q_proj","k_proj","v_proj","o_proj",
#         "gate_proj","up_proj","down_proj"
#     ],
# )

# print("LoRA config loaded.")


# # ============================================================================
# # STEP 6: TRAINING ARGUMENTS
# # ============================================================================

# print("\n" + "="*80)
# print("STEP 6: TRAINING ARGS")
# print("="*80)

# training_args = TrainingArguments(
#     output_dir="clarifier_sft_output",
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=8,
#     num_train_epochs=1,
#     learning_rate=2e-4,
#     warmup_ratio=0.05,
#     bf16=True,
#     logging_steps=10,
#     save_steps=200,
#     save_total_limit=2,
#     optim="paged_adamw_8bit",
#     report_to="none",
# )


# # ============================================================================
# # STEP 7: CREATE TRAINER
# # ============================================================================

# print("\n" + "="*80)
# print("STEP 7: INITIALIZING TRAINER")
# print("="*80)

# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=formatted_dataset,
#     dataset_text_field="text",   # IMPORTANT
#     max_seq_length=2048,
#     packing=True,
#     peft_config=lora_config,
#     args=training_args,
# )

# print("Trainer initialized successfully.")


# # ============================================================================
# # STEP 8: TRAIN
# # ============================================================================

# print("\n" + "="*80)
# print("STEP 8: TRAINING")
# print("="*80)

# trainer.train()

# print("Training complete!")


# # ============================================================================
# # STEP 9: SAVE MODEL
# # ============================================================================

# trainer.save_model("clarifier_final_model")
# tokenizer.save_pretrained("clarifier_final_model")

# print("Model saved!")


# # ============================================================================
# # STEP 10: TEST INFERENCE
# # ============================================================================

# print("\n" + "="*80)
# print("STEP 10: TESTING INFERENCE")
# print("="*80)

# from peft import AutoPeftModelForCausalLM

# trained_model = AutoPeftModelForCausalLM.from_pretrained(
#     "clarifier_final_model",
#     torch_dtype=torch.bfloat16,
#     device_map="auto"
# )

# test_question = "What is the best programming language?"

# system_msg = (
#     "You are a question understanding agent. Decide ambiguity, explain reasoning, "
#     "list facets if needed, and respond in required format."
# )

# messages = [
#     {"role": "system", "content": system_msg},
#     {"role": "user", "content": test_question}
# ]

# input_text = tokenizer.apply_chat_template(
#     messages, tokenize=False, add_generation_prompt=True
# )

# inputs = tokenizer(input_text, return_tensors="pt").to(trained_model.device)

# with torch.no_grad():
#     outputs = trained_model.generate(
#         **inputs,
#         max_new_tokens=300,
#         temperature=0.7,
#         top_p=0.9,
#         do_sample=True
#     )

# print("\n--- MODEL RESPONSE ---")
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))
# print("-----------------------")


In [ ]:
# sample_out = run_model_on_question("Who won the US Open?")
# print(sample_out)

In [ ]:
# print(run_model_on_question("Who is the president?"))

In [ ]:
# import pandas as pd

# final_sft_df  # <-- adjust filename
# print(final_sft_df.head())
# print(len(final_sft_df), "rows loaded")

In [ ]:
# import torch
# from transformers import AutoTokenizer
# from peft import AutoPeftModelForCausalLM

# # ============================================================
# # LOAD MODEL
# # ============================================================
# model_path = "clarifier_final_model"

# tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
# model = AutoPeftModelForCausalLM.from_pretrained(
#     model_path,
#     torch_dtype=torch.bfloat16,
#     device_map="auto"
# )

# model.eval()

# # ============================================================
# # INFERENCE FUNCTION
# # ============================================================
# def run_inference(question):

#     system_prompt = (
#         "You are a question understanding agent. For each user question:\n"
#         "1) Decide if it is ambiguous.\n"
#         "2) Explain your reasoning.\n"
#         "3) List facets if ambiguous.\n"
#         "4) Ask a clarifying question OR give a direct answer.\n\n"
#         "Format:\n"
#         "Action: Clarify|Answer\n"
#         "Reasoning: ...\n"
#         "Facets: [...]\n"
#         "Response: ...\n"
#     )

#     messages = [
#         {"role": "system", "content": system_prompt},
#         {"role": "user", "content": question}
#     ]

#     # Build input according to Qwen chat template
#     prompt = tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True
#     )

#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=300,
#             temperature=0.7,
#             top_p=0.9,
#             do_sample=True
#         )

#     decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

#     # ============================================================
#     # EXTRACT ASSISTANT OUTPUT ONLY
#     # ============================================================
#     if "<|im_start|>assistant" in decoded:
#         assistant = decoded.split("<|im_start|>assistant")[1]
#     else:
#         assistant = decoded

#     assistant = assistant.split("<|im_end|>")[0].strip()

#     return assistant


# # ================== TEST ==================
# print(run_inference("Who won the US Open?"))


In [ ]:
# import pandas as pd
# import torch
# from transformers import AutoTokenizer
# from peft import AutoPeftModelForCausalLM

# # ============================================================
# # LOAD MODEL
# # ============================================================
# model_path = "clarifier_final_model"

# tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
# model = AutoPeftModelForCausalLM.from_pretrained(
#     model_path,
#     torch_dtype=torch.bfloat16,
#     device_map="auto"
# )

# model.eval()

# # ============================================================
# # INFERENCE (MATCHES TRAINING EXACTLY)
# # ============================================================
# def run_inference(question):

#     system_prompt = (
#         "You are a question understanding agent. For each user question:\n"
#         "1) Decide if it is ambiguous.\n"
#         "2) Explain your reasoning.\n"
#         "3) List facets if ambiguous.\n"
#         "4) Ask a clarifying question OR give a direct answer.\n\n"
#         "Format:\n"
#         "Action: Clarify|Answer\n"
#         "Reasoning: ...\n"
#         "Facets: [...]\n"
#         "Response: ...\n"
#     )

#     messages = [
#         {"role": "system", "content": system_prompt},
#         {"role": "user", "content": question}
#     ]

#     prompt = tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True
#     )

#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=200,
#             do_sample=False,        # <---- FIXED (GREEDY)
#         )

#     decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

#     # Extract only assistant part
#     if "<|im_start|>assistant" in decoded:
#         assistant = decoded.split("<|im_start|>assistant")[1]
#     else:
#         assistant = decoded

#     assistant = assistant.split("<|im_end|>")[0].strip()

#     return assistant


# # ============================================================
# # EVALUATION
# # ============================================================

# def extract_action(text):
#     for line in text.split("\n"):
#         if line.lower().startswith("action:"):
#             if "clarify" in line.lower():
#                 return "Clarify"
#             if "answer" in line.lower():
#                 return "Answer"
#     return None

# # SAMPLE EVAL DATASET
# eval_df = final_sft_df.sample(200, random_state=42)

# correct = 0
# total = len(eval_df)
# errors = []

# print("\n🔍 Running evaluation...\n")

# for _, row in eval_df.iterrows():
#     q = row["question"]
#     true = row["action"]

#     out = run_inference(q)
#     pred = extract_action(out)

#     if pred == true:
#         correct += 1
#     else:
#         errors.append((q, true, pred, out))

# acc = correct / total

# print("===== RESULTS =====")
# print(f"Action accuracy: {acc:.3f}\n")

# print("===== SAMPLE ERRORS =====")
# for q, true, pred, out in errors[:5]:
#     print("------------------------------------------------------------")
#     print(f"Q: {q}")
#     print(f"True: {true}")
#     print(f"Pred: {pred}")
#     print("\nMODEL OUTPUT:\n", out[:500])


# RE-DOING SFT PT 2

In [ ]:
# ============================================================================
# QWEN SFT TRAINING PIPELINE (AMBIGQA STYLE, FIXED)
# ============================================================================

import json
import ast
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

print("\n" + "="*80)
print("STEP 1: PREPARING DATASET")
print("="*80)

# Use your dataframe
df = final_sft_df.copy()

# ----------------------------------------------------------
# Cleaning helpers
# ----------------------------------------------------------

def clean_facets(facets):
    if isinstance(facets, list):
        return facets
    if isinstance(facets, str):
        try:
            val = ast.literal_eval(facets)
            return val if isinstance(val, list) else []
        except:
            return []
    return []

def clean_response(resp):
    if resp is None:
        return ""
    if isinstance(resp, list):
        return resp[0] if resp else ""
    if isinstance(resp, str) and resp.startswith("[") and resp.endswith("]"):
        try:
            lst = ast.literal_eval(resp)
            return lst[0] if lst else ""
        except:
            pass
    return str(resp).strip()

# ----------------------------------------------------------
# Build SFT JSONL (instruction/input/output is just an intermediate form)
# ----------------------------------------------------------

records = []

for _, row in df.iterrows():
    q = row["question"]
    is_amb = row["is_ambiguous"]
    reasoning = row["reasoning"]
    action = row["action"]           # "Clarify" or "Answer"
    facets = clean_facets(row["facets"])
    response = clean_response(row["positive_response"])

    records.append({
        "instruction": (
            "Decide whether the user's question is ambiguous (in the AmbigQA sense: "
            "multiple plausible interpretations or answer targets). "
            "If ambiguous, explain the ambiguity, list facets, and ask a clarifying question. "
            "If not ambiguous, explain why and answer directly."
        ),
        "input": q,
        "output": {
            "action": action,
            "reasoning": reasoning,
            "facets": facets if is_amb else [],
            "response": response
        }
    })

with open("sft_data.jsonl", "w") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

print(f"✅ Created SFT dataset with {len(records)} rows")


# ============================================================================
# STEP 2: LOAD DATASET
# ============================================================================

print("\n" + "="*80)
print("STEP 2: LOADING DATASET")
print("="*80)

train_dataset = load_dataset("json", data_files="sft_data.jsonl")["train"]
print(f"Loaded {len(train_dataset)} examples")


# ============================================================================
# STEP 3: LOAD MODEL + TOKENIZER
# ============================================================================

print("\n" + "="*80)
print("STEP 3: LOADING MODEL & TOKENIZER")
print("="*80)

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

# Critical fix
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

print("✅ Model & tokenizer loaded.")


# ============================================================================
# STEP 4: FORMAT TEXT FOR TRAINING USING CHAT TEMPLATE
# ============================================================================

print("\n" + "="*80)
print("STEP 4: FORMATTING DATASET (CHAT TEMPLATE)")
print("="*80)

def format_chat(example):
    question  = example["input"]
    action    = example["output"]["action"]
    reasoning = example["output"]["reasoning"]
    facets    = example["output"]["facets"]
    response  = example["output"]["response"]

    facets_str = "[" + ", ".join(facets) + "]" if facets else "[]"

    # System message: explicitly references AmbigQA-style ambiguity
    system_msg = (
        "You are a question understanding agent trained on AmbigQA-style ambiguity. "
        "For each user question:\n"
        "1) Decide if it is ambiguous (i.e., multiple plausible interpretations/answers).\n"
        "2) Explain your reasoning.\n"
        "3) List facets if ambiguous.\n"
        "4) Ask a clarifying question OR give a direct answer.\n\n"
        "Format:\n"
        "Action: Clarify|Answer\n"
        "Reasoning: ...\n"
        "Facets: [...]\n"
        "Response: ...\n"
    )

    assistant_msg = (
        f"Action: {action}\n"
        f"Reasoning: {reasoning}\n"
        f"Facets: {facets_str}\n"
        f"Response: {response}"
    )

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": question},
        {"role": "assistant", "content": assistant_msg}
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": formatted}

formatted_dataset = train_dataset.map(
    format_chat,
    remove_columns=train_dataset.column_names
)

print("Example formatted text:")
print("-"*80)
print(formatted_dataset[0]["text"][:400])
print("-"*80)


# ============================================================================
# STEP 5: LoRA CONFIG
# ============================================================================

print("\n" + "="*80)
print("STEP 5: LORA CONFIG")
print("="*80)

lora_config = LoraConfig(
    r=32,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
)

print("LoRA config loaded.")


# ============================================================================
# STEP 6: TRAINING ARGUMENTS
# ============================================================================

print("\n" + "="*80)
print("STEP 6: TRAINING ARGS")
print("="*80)

training_args = TrainingArguments(
    output_dir="clarifier_sft_output_ambigqa",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,        # 🔥 more than 1 epoch now
    learning_rate=2e-4,
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    report_to="none",
)

print("Training args ready.")


# ============================================================================
# STEP 7: CREATE TRAINER
# ============================================================================

print("\n" + "="*80)
print("STEP 7: INITIALIZING TRAINER")
print("="*80)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,           # 🔥 CRITICAL FIX: NO PACKING
    peft_config=lora_config,
    args=training_args,
)

print("Trainer initialized successfully.")


# ============================================================================
# STEP 8: TRAIN
# ============================================================================

print("\n" + "="*80)
print("STEP 8: TRAINING")
print("="*80)

trainer.train()

print("Training complete!")


# ============================================================================
# STEP 9: SAVE MODEL
# ============================================================================

save_path = "clarifier_final_model_ambigqa"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to: {save_path}")


STEP 1: PREPARING DATASET
✅ Created SFT dataset with 10036 rows

STEP 2: LOADING DATASET


Generating train split: 0 examples [00:00, ? examples/s]

Loaded 10036 examples

STEP 3: LOADING MODEL & TOKENIZER


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model & tokenizer loaded.

STEP 4: FORMATTING DATASET (CHAT TEMPLATE)


...

Example formatted text:
--------------------------------------------------------------------------------
<|im_start|>system
You are a question understanding agent trained on AmbigQA-style ambiguity. For each user question:
1) Decide if it is ambiguous (i.e., multiple plausible interpretations/answers).
2) Explain your reasoning.
3) List facets if ambiguous.
4) Ask a clarifying question OR give a direct answer.

Format:
Action: Clarify|Answer
Reasoning: ...
Facets: [...]
Response: ...
<|im_end|>
<|im_
--------------------------------------------------------------------------------

STEP 5: LORA CONFIG
LoRA config loaded.

STEP 6: TRAINING ARGS
Training args ready.

STEP 7: INITIALIZING TRAINER


Map:   0%|          | 0/10036 [00:00<?, ? examples/s]

/home/bsanghi2/.local/lib/python3.11/site-packages/trl/trainer/sft_trainer.py:323: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainer initialized successfully.

STEP 8: TRAINING


Step,Training Loss
10,2.982600
20,2.800100
30,2.020300
40,1.078800
50,0.605100
60,0.489600
70,0.417100
80,0.434000
90,0.402400
100,0.373300


In [ ]:
# ============================================================
# MULTI-CHECKPOINT EVALUATOR FOR QWEN CLARIFIER MODEL
# ============================================================

import os
import re
import pandas as pd
import torch
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM

# ============================================================
# CONFIG
# ============================================================

CHECKPOINT_DIR = "clarifier_sft_output_ambigqa"
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# SAMPLE 500 rows for evaluation to keep it fast
EVAL_SIZE = 50

print("🔧 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ============================================================
# Load your eval dataframe
# (Assumes final_sft_df exists in session)
# ============================================================

df_eval = final_sft_df.copy()
df_eval = df_eval.sample(min(EVAL_SIZE, len(df_eval))).reset_index(drop=True)

print(f"📊 Using {len(df_eval)} evaluation samples")

# ============================================================
# Parse model output action
# ============================================================

def extract_action(text):
    """
    Extracts `Clarify` or `Answer` from model output.
    Looks for lines like: "Action: Clarify"
    """
    text = text.strip()

    # Search explicitly in formatted lines
    for line in text.split("\n"):
        if line.lower().startswith("action:"):
            action = line.split(":", 1)[1].strip()
            if action in ["Clarify", "Answer"]:
                return action

    return None


# ============================================================
# Run inference on a single question for any model
# ============================================================

SYSTEM_PROMPT = (
    "You are a question understanding agent.\n"
    "1) Decide if a question is ambiguous.\n"
    "2) Explain your reasoning.\n"
    "3) List facets (if ambiguous).\n"
    "4) Output in format:\n\n"
    "Action: Clarify|Answer\n"
    "Reasoning: ...\n"
    "Facets: [...]\n"
    "Response: ...\n"
)

def run_inference(model, question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            temperature=None,
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=False)

    # Extract assistant section
    if "<|im_start|>assistant" in decoded:
        return decoded.split("<|im_start|>assistant")[-1].strip()

    return decoded.strip()


# ============================================================
# Evaluate a checkpoint directory
# ============================================================

def evaluate_checkpoint(path):
    print(f"\n🚀 Loading model: {path}")

    try:
        model = AutoPeftModelForCausalLM.from_pretrained(
            path,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )
    except Exception as e:
        print(f"❌ Failed to load {path}: {e}")
        return None

    correct = 0
    total = len(df_eval)
    errors = []

    for idx, row in df_eval.iterrows():
        q = row["question"]
        true_label = row["action"]

        model_output = run_inference(model, q)
        pred_label = extract_action(model_output)

        if pred_label == true_label:
            correct += 1
        else:
            errors.append((q, true_label, pred_label, model_output))

    accuracy = correct / total
    return accuracy, errors


# ============================================================
# Find all checkpoints
# ============================================================

print("\n🔍 Searching for checkpoints...")
checkpoints = []

for item in os.listdir(CHECKPOINT_DIR):
    if item.startswith("checkpoint-"):
        num = int(item.split("-")[-1])
        checkpoints.append((num, os.path.join(CHECKPOINT_DIR, item)))

checkpoints = sorted(checkpoints)

print(f"📦 Found checkpoints: {[num for num, _ in checkpoints]}")

# ============================================================
# Evaluate each checkpoint
# ============================================================

results = []

for num, path in checkpoints:
    acc, errors = evaluate_checkpoint(path)
    if acc is not None:
        results.append((num, acc, errors))
        print(f"✅ checkpoint-{num} accuracy: {acc:.3f}")

# ============================================================
# Leaderboard
# ============================================================

print("\n🏆 CHECKPOINT LEADERBOARD\n")
for num, acc, _ in sorted(results, key=lambda x: -x[1]):
    print(f"checkpoint-{num}: {acc:.3f}")

# ============================================================
# Show sample errors from the best model
# ============================================================

best_num, best_acc, best_errors = sorted(results, key=lambda x: -x[1])[0]

print("\n===== SAMPLE ERRORS FROM BEST MODEL =====")
print(f"📌 Best checkpoint: {best_num}  (accuracy={best_acc:.3f})")

for i, (q, true, pred, out) in enumerate(best_errors[:5]):
    print("\n------------------------------------------------------------")
    print(f"Q: {q}")
    print(f"True: {true}")
    print(f"Pred: {pred}")
    print("\nMODEL OUTPUT:\n", out[:500])


🔧 Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


📊 Using 50 evaluation samples

🔍 Searching for checkpoints...
📦 Found checkpoints: [2400, 2510]

🚀 Loading model: clarifier_sft_output_ambigqa/checkpoint-2400


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ checkpoint-2400 accuracy: 0.560

🚀 Loading model: clarifier_sft_output_ambigqa/checkpoint-2510


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ checkpoint-2510 accuracy: 0.520

🏆 CHECKPOINT LEADERBOARD

checkpoint-2400: 0.560
checkpoint-2510: 0.520

===== SAMPLE ERRORS FROM BEST MODEL =====
📌 Best checkpoint: 2400  (accuracy=0.560)

------------------------------------------------------------
Q: Who is girl in we don't talk anymore?
True: Clarify
Pred: Answer

MODEL OUTPUT:
 Action: Answer
Reasoning: The question is not ambiguous because it specifies the song "We Don't Talk Anymore" and asks about a particular character within that context. This information alone is sufficient to uniquely identify the subject being referred to, as the song has a known lyric with a specific girl mentioned.
Facets: []
Response: Taylor Swift<|im_end|>

------------------------------------------------------------
Q: When did one direction go on a hiatus?
True: Answer
Pred: Clarify

MODEL OUTPUT:
 Action: Clarify
Reasoning: The question is ambiguous because it does not specify which phase of One Direction's hiatus is being referred to. Without th

# training one more epoch

In [ ]:
import json
import ast
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer

# ----------------------------------------
# LOAD YOUR ORIGINAL DATAFRAME
# ----------------------------------------
df = final_sft_df.copy()

def clean_facets(f):
    if isinstance(f, list):
        return f
    if isinstance(f, str):
        try:
            v = ast.literal_eval(f)
            return v if isinstance(v, list) else []
        except:
            return []
    return []

def clean_response(r):
    if r is None:
        return ""
    if isinstance(r, list):
        return r[0] if r else ""
    if isinstance(r, str) and r.startswith("[") and r.endswith("]"):
        try:
            lst = ast.literal_eval(r)
            return lst[0] if lst else ""
        except:
            pass
    return str(r).strip()

# ----------------------------------------
# LOAD TOKENIZER (needed for formatting)
# ----------------------------------------
model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ----------------------------------------
# BUILD FORMATTED DATASET
# ----------------------------------------
def format_chat_row(row):
    q = row["question"]
    action = row["action"]
    reasoning = row["reasoning"]
    facets = clean_facets(row["facets"])
    response = clean_response(row["positive_response"])

    facets_str = "[" + ", ".join(facets) + "]" if facets else "[]"

    system_msg = (
        "You are a question understanding agent. "
        "For each user question:\n"
        "1) Decide if it is ambiguous.\n"
        "2) Explain your reasoning.\n"
        "3) List facets if ambiguous.\n"
        "4) Ask a clarifying question OR give a direct answer.\n\n"
        "Format:\n"
        "Action: Clarify|Answer\n"
        "Reasoning: ...\n"
        "Facets: [...]\n"
        "Response: ...\n"
    )

    assistant_msg = (
        f"Action: {action}\n"
        f"Reasoning: {reasoning}\n"
        f"Facets: {facets_str}\n"
        f"Response: {response}"
    )

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": q},
        {"role": "assistant", "content": assistant_msg}
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return formatted

df["text"] = df.apply(format_chat_row, axis=1)

# Convert to HF Dataset
formatted_dataset = Dataset.from_pandas(df[["text"]])

print("Formatted dataset size:", len(formatted_dataset))
print("Example:\n", formatted_dataset[0]["text"][:500])

# Save if you want
formatted_dataset.save_to_disk("formatted_sft_dataset")

Formatted dataset size: 10036
Example:
 <|im_start|>system
You are a question understanding agent. For each user question:
1) Decide if it is ambiguous.
2) Explain your reasoning.
3) List facets if ambiguous.
4) Ask a clarifying question OR give a direct answer.

Format:
Action: Clarify|Answer
Reasoning: ...
Facets: [...]
Response: ...
<|im_end|>
<|im_start|>user
When did the simpsons first air on television?<|im_end|>
<|im_start|>assistant
Action: Clarify
Reasoning: The question is ambiguous because it does not specify whether 'The S


Saving the dataset (0/1 shards):   0%|          | 0/10036 [00:00<?, ? examples/s]

In [ ]:
# ============================================================
# RESUME QWEN SFT TRAINING — EPOCH 2 USING CHECKPOINT 2400
# (BUG-FREE, MATCHES YOUR ORIGINAL WORKING PIPELINE)
# ============================================================

import json
import ast
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, TrainingArguments
from peft import AutoPeftModelForCausalLM
from trl import SFTTrainer

print("\n==================== STEP 0: LOAD ORIGINAL DATA ====================")

df = final_sft_df.copy()

# ---- Helpers used in your original epoch 1 script ----
def clean_facets(f):
    if isinstance(f, list):
        return f
    if isinstance(f, str):
        try:
            v = ast.literal_eval(f)
            return v if isinstance(v, list) else []
        except:
            return []
    return []

def clean_response(r):
    if r is None:
        return ""
    if isinstance(r, list):
        return r[0] if r else ""
    if isinstance(r, str) and r.startswith("[") and r.endswith("]"):
        try:
            lst = ast.literal_eval(r)
            return lst[0] if lst else ""
        except:
            pass
    return str(r).strip()

print("Loaded dataframe with", len(df), "rows.")


# ============================================================
# STEP 1: LOAD TOKENIZER
# ============================================================

print("\n==================== STEP 1: LOAD TOKENIZER ====================")

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Tokenizer ready.")


# ============================================================
# STEP 2: REBUILD FORMATTED DATASET (IDENTICAL TO EPOCH 1)
# ============================================================

print("\n==================== STEP 2: REBUILD FORMATTED DATASET ====================")

def format_chat_row(row):
    q = row["question"]
    action = row["action"]
    reasoning = row["reasoning"]
    facets = clean_facets(row["facets"])
    response = clean_response(row["positive_response"])

    facets_str = "[" + ", ".join(facets) + "]" if facets else "[]"

    system_msg = (
        "You are a question understanding agent trained on AmbigQA-style ambiguity. "
        "For each user question:\n"
        "1) Decide if it is ambiguous.\n"
        "2) Explain your reasoning.\n"
        "3) List facets if ambiguous.\n"
        "4) Ask a clarifying question OR give a direct answer.\n\n"
        "Format:\n"
        "Action: Clarify|Answer\n"
        "Reasoning: ...\n"
        "Facets: [...]\n"
        "Response: ...\n"
    )

    assistant_msg = (
        f"Action: {action}\n"
        f"Reasoning: {reasoning}\n"
        f"Facets: {facets_str}\n"
        f"Response: {response}"
    )

    msgs = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": q},
        {"role": "assistant", "content": assistant_msg}
    ]

    return tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=False
    )

df["text"] = df.apply(format_chat_row, axis=1)

formatted_dataset = Dataset.from_pandas(df[["text"]])

print("Formatted dataset created with size:", len(formatted_dataset))
print("Example:\n", formatted_dataset[0]["text"][:300])


# ============================================================
# STEP 3: LOAD CHECKPOINT-2400 (THE RIGHT WAY)
# ============================================================

print("\n==================== STEP 3: LOAD CHECKPOINT 2400 ====================")

checkpoint_path = "clarifier_sft_output_ambigqa/checkpoint-2400"

model = AutoPeftModelForCausalLM.from_pretrained(
    checkpoint_path,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.config.use_cache = False

print("Loaded PEFT LoRA model from checkpoint-2400.")


# Make sure LoRA parameters require grad
trainable_params = 0
total_params = 0
for name, param in model.named_parameters():
    total_params += param.numel()
    if "lora_" in name:
        param.requires_grad = True
        trainable_params += param.numel()
    else:
        # keep base model frozen (8-bit) to avoid exploding VRAM
        param.requires_grad = False

print(f"Trainable LoRA params: {trainable_params} / {total_params} "
      f"({trainable_params/total_params:.4%})")


# ============================================================
# STEP 4: TRAINING ARGS FOR SECOND EPOCH
# ============================================================

print("\n==================== STEP 4: SET TRAINING ARGS ====================")

training_args = TrainingArguments(
    output_dir="clarifier_sft_output_ambigqa_epoch2",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,        # <-- SECOND EPOCH
    learning_rate=2e-4,
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    report_to="none",
)

print("Training args ready.")


# ============================================================
# STEP 5: CREATE TRAINER (IMPORTANT: NO peft_config)
# ============================================================

print("\n==================== STEP 5: INITIALIZE TRAINER ====================")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,   # same as epoch 1
    args=training_args,
)

print("Trainer ready.")


# ============================================================
# STEP 6: TRAIN FOR SECOND EPOCH
# ============================================================

print("\n==================== STEP 6: TRAINING EPOCH 2 ====================")

trainer.train(resume_from_checkpoint=False)

print("🎉 Finished epoch 2!")


# ============================================================
# STEP 7: SAVE FINAL MODEL
# ============================================================

save_path = "clarifier_final_model_ambigqa_epoch2"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Saved final model to:", save_path)


==================== STEP 0: LOAD ORIGINAL DATA ====================
Loaded dataframe with 10036 rows.

==================== STEP 1: LOAD TOKENIZER ====================
Tokenizer ready.

==================== STEP 2: REBUILD FORMATTED DATASET ====================
Formatted dataset created with size: 10036
Example:
 <|im_start|>system
You are a question understanding agent trained on AmbigQA-style ambiguity. For each user question:
1) Decide if it is ambiguous.
2) Explain your reasoning.
3) List facets if ambiguous.
4) Ask a clarifying question OR give a direct answer.

Format:
Action: Clarify|Answer
Reasoning:

==================== STEP 3: LOAD CHECKPOINT 2400 ====================


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded PEFT LoRA model from checkpoint-2400.
Trainable LoRA params: 80740352 / 7696356864 (1.0491%)

==================== STEP 4: SET TRAINING ARGS ====================
Training args ready.

==================== STEP 5: INITIALIZE TRAINER ====================


Map:   0%|          | 0/10036 [00:00<?, ? examples/s]

/home/bsanghi2/.local/lib/python3.11/site-packages/trl/trainer/sft_trainer.py:323: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainer ready.

==================== STEP 6: TRAINING EPOCH 2 ====================


Step,Training Loss
10,0.319800
20,0.262400
30,0.292400
40,0.263000
50,0.293100
60,0.262100
70,0.262800
80,0.288700
90,0.283700
100,0.254300


🎉 Finished epoch 2!
Saved final model to: clarifier_final_model_ambigqa_epoch2


In [ ]:
# ============================================================
# FINAL EVALUATION SCRIPT FOR AMBIGQA CLARIFIER MODEL
# ============================================================

import re
import pandas as pd
import torch
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM
from difflib import SequenceMatcher

# ============================================================
# CONFIG
# ============================================================

MODEL_PATH = "clarifier_final_model_ambigqa_epoch2"   # <<< your saved model
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
EVAL_SIZE = 500      # evaluate on 500 rows
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("🔧 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("🔧 Loading model...")
model = AutoPeftModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# ============================================================
# LOAD DATA
# (final_sft_df must exist in memory)
# ============================================================

df_eval = final_sft_df.copy()
df_eval = df_eval.sample(min(EVAL_SIZE, len(df_eval))).reset_index(drop=True)

print(f"📊 Evaluating on {len(df_eval)} samples\n")


# ============================================================
# HELPER: CLEAN MODEL OUTPUT TO EXTRACT ASSISTANT PART ONLY
# ============================================================

def extract_assistant(text):
    """Extract assistant message content cleanly."""
    if "<|im_start|>assistant" in text:
        text = text.split("<|im_start|>assistant")[-1]

    # Remove final <|im_end|> tokens
    text = text.replace("<|im_end|>", "").strip()

    # Remove accidental system/user echoes
        # Remove accidental system/user echoes
    cleaned_lines = []
    for line in text.split("\n"):
        if line.strip().lower().startswith("system"):
            continue
        if line.strip().lower().startswith("user"):
            continue
        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines).strip()
    return text


# ============================================================
# EXTRACT PREDICTED ACTION
# ============================================================

def extract_action(text):
    """Find 'Action: Clarify' or 'Action: Answer'."""
    for line in text.split("\n"):
        if line.lower().startswith("action:"):
            val = line.split(":", 1)[1].strip()
            if val in ["Clarify", "Answer"]:
                return val
    return None


# ============================================================
# EXTRACT PREDICTED ANSWER (if Action = Answer)
# ============================================================

def extract_response(text):
    """Get the content of 'Response:'"""
    for line in text.split("\n"):
        if line.lower().startswith("response:"):
            return line.split(":", 1)[1].strip()
    return ""


# ============================================================
# SEMANTIC STRING MATCHING FOR ANSWER CHECKING
# ============================================================

def semantic_match(a, b):
    """
    More lenient answer comparison:
    - lowercase
    - remove punctuation
    - fuzzy token match
    """
    if not a or not b:
        return False

    import string
    table = str.maketrans("", "", string.punctuation)

    a_clean = a.lower().translate(table).strip()
    b_clean = b.lower().translate(table).strip()

    if a_clean == b_clean:
        return True

    # fuzzy match ratio
    ratio = SequenceMatcher(None, a_clean, b_clean).ratio()
    return ratio > 0.75


# ============================================================
# RUN INFERENCE
# ============================================================

SYSTEM_PROMPT = (
    "You are a question understanding agent.\n"
    "1) Decide if a question is ambiguous.\n"
    "2) Explain your reasoning.\n"
    "3) List facets.\n"
    "4) Output ONLY in the format:\n\n"
    "Action: Clarify|Answer\n"
    "Reasoning: ...\n"
    "Facets: [...]\n"
    "Response: ...\n"
)

def run_inference(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=False)
    return extract_assistant(decoded)


# ============================================================
# EVALUATION LOOP
# ============================================================

correct_action = 0
correct_answer = 0
answer_cases = 0

errors = []

for _, row in df_eval.iterrows():
    q = row["question"]
    gold_action = row["action"]
    gold_answer = row["positive_response"]

    model_out = run_inference(q)
    pred_action = extract_action(model_out)
    pred_answer = extract_response(model_out)

    # Action accuracy
    if pred_action == gold_action:
        correct_action += 1
    else:
        errors.append((q, gold_action, pred_action, model_out))

    # Answer accuracy (semantic)
    if gold_action == "Answer":
        answer_cases += 1
        if semantic_match(pred_answer, gold_answer):
            correct_answer += 1


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n===== RESULTS =====")
print(f"Action accuracy: {correct_action / len(df_eval):.3f}")

if answer_cases > 0:
    print(f"Answer accuracy (semantic): {correct_answer / answer_cases:.3f}")
else:
    print("No Answer examples in evaluation subset.")

print("\n===== SAMPLE ERRORS =====")
for q, gold, pred, out in errors[:5]:
    print("\n------------------------------------------------------------")
    print(f"Q: {q}")
    print(f"Gold Action: {gold}")
    print(f"Pred Action: {pred}")
    print(f"MODEL OUTPUT:\n{out[:500]}")

🔧 Loading tokenizer...
🔧 Loading model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

📊 Evaluating on 500 samples


===== RESULTS =====
Action accuracy: 0.600
Answer accuracy (semantic): 0.238

===== SAMPLE ERRORS =====

------------------------------------------------------------
Q: When can the new ball be taken in test cricket?
Gold Action: Clarify
Pred Action: Answer
MODEL OUTPUT:
Action: Answer
Reasoning: The question is not ambiguous because it specifies the sport (test cricket) and asks about a particular rule (when the new ball can be taken). This information alone is sufficient to determine the answer based on the rules of test cricket.
Facets: [Sport Type, Event Category]
Response: In test cricket, the new ball can be taken at the start of each innings or if the ball becomes worn out during play.

------------------------------------------------------------
Q: Metal gear solid v ground zeroes how many missions?
Gold Action: Clarify
Pred Action: Answer
MODEL OUTPUT:
Action: Answer
Reasoning: The question is not ambiguous because it specifies the game "Metal Gea

# DPO PIPELINE

### merge the negative and postive dataset

In [ ]:
import pandas as pd
final_sft_df=pd.read_csv('final_SFT_data_for_use.csv')
final_sft_df.head()

,Unnamed: 0,question,is_ambiguous,facets,reasoning,positive_response,action
0,0,When did the simpsons first air on television?,True,"['Type of Show', 'Platform']",The question is ambiguous because it does not ...,Are you asking about the original TV series or...,Clarify
1,1,Who played george washington in the john adams...,False,[],The question is not ambiguous because it speci...,['David Morse'],Answer
2,2,What is the legal age of marriage in usa?,True,"['State', 'Circumstances for Marriage']",The question is ambiguous because the legal ag...,In which U.S. state and under what circumstanc...,Clarify
3,3,Who starred in barefoot in the park on broadway?,True,['Character Name'],The question is ambiguous because it does not ...,Did you mean the character names or the actors...,Clarify
4,4,When did the manhattan project began and end?,True,['Time Frame Interpretation'],The question is ambiguous because it does not ...,In which specific year or time period are you ...,Clarify


In [ ]:
neg_data=pd.read_csv('neg_synthetic_full.csv')
neg_data.head()

,Unnamed: 0,question,type,qaPairs,disambig_questions,is_ambiguous,facets,neg_response
0,0,When did the simpsons first air on television?,multipleQAs,[{'question': ['When did the Simpsons first ai...,['When did the Simpsons first air on televisio...,True,"['Type of Show', 'Platform']","The Simpsons first aired on December 17, 1989."
1,1,Who played george washington in the john adams...,singleAnswer,"[{'question': [], 'answer': []}]",[],False,[],Could you tell me which season of the John Ada...
2,2,What is the legal age of marriage in usa?,multipleQAs,[{'question': ['What is the legal age of marri...,"['What is the legal age of marriage, without p...",True,"['State', 'Circumstances for Marriage']",The legal age of marriage in the USA is 18.
3,3,Who starred in barefoot in the park on broadway?,multipleQAs,[{'question': ['Who starred in barefoot in the...,['Who starred in barefoot in the park on broad...,True,['Character Name'],Walter Matthau and Anne Bancroft starred in Ba...
4,4,When did the manhattan project began and end?,multipleQAs,[{'question': ['Based on the initial thoughts ...,['Based on the initial thoughts of the project...,True,['Time Frame Interpretation'],The Manhattan Project began in 1942 and ended ...


In [ ]:
final_dpo_df=pd.merge(final_sft_df,neg_data[['question', 'neg_response']], how='left', on='question')
final_dpo_df.to_csv('final_dpo_df.csv')

# Starting on DPO- Bhoomika

In [ ]:
!pip install torch

  Using cached torch-2.9.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cusolver_cu12-11.7.3.90-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cusparse_cu12-12.5.8.93-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.8 kB)
Using cached torch-2.9.1-cp311-cp311-manylinux_2_28_x86_64.whl (899.8 MB)
Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl (706.8 MB)
Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (193.1 MB)
Using cached nvidia_cusolver_cu12-11.7.3.90-py3-none-manylinux_2_27_x86_64.whl (267.5 MB)
Using cached nvidia_cusparse_cu12-12.5.8.93-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (288.2 MB)


In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers>=4.40.0 datasets accelerate trl>=0.9.0 peft>=0.11.0 bitsandbytes pandas scikit-learn

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 71.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 24.4 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 88.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 5.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 8.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 20.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 43.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 23.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 15.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147

In [ ]:
!nvidia-smi

Tue Nov 25 20:19:05 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:00:05.0 Off |                    0 |
| N/A   26C    P0             82W /  500W |   21924MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install transformers>=4.40.0 datasets accelerate trl>=0.9.0 peft>=0.11.0 bitsandbytes pandas scikit-learn

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 25.2 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 83.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 6.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 10.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 17.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 46.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 26.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 18.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 18.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# DPO- Jupyter NCSA- A100 80Gb

In [ ]:
!pip install transformers>=4.45.0 trl>=0.12.0 --upgrade

In [ ]:
!which gcc
!gcc --version

/bin/bash: line 1: gcc: command not found


In [ ]:
import os
os.environ['CC'] = '/usr/bin/gcc'
os.environ['CXX'] = '/usr/bin/g++'

In [ ]:
!ls /usr/bin/ | grep -E "gcc|cc|clang"

kpseaccess
run-one-until-success


In [ ]:
!which cc
!which clang
!ls /opt/*/bin/ 2>/dev/null | head -20
!cat /etc/os-release

2to3
2to3-3.11
accelerate
accelerate-config
accelerate-estimate-memory
accelerate-launch
accelerate-merge-weights
activate
adig
ahost
alembic
bsdcat
bsdcpio
bsdtar
bsdunzip
bunzip2
bzcat
bzcmp
bzdiff
bzegrep
PRETTY_NAME="Ubuntu 22.04.3 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.3 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy


In [ ]:
!conda install -y gcc_linux-64 gxx_linux-64/

Solving environment: unsuccessful initial attempt using frozen solve. Retrying with flexible solve.
Solving environment: unsuccessful initial attempt using frozen solve. Retrying with flexible solve.

PackagesNotFoundError: The following packages are not available from current channels:

  - gxx_linux-64/

Current channels:

  - https://conda.anaconda.org/conda-forge/linux-64
  - https://conda.anaconda.org/conda-forge/noarch

To search for alternate channels that may provide the conda package you're
looking for, navigate to

    https://anaconda.org

and use the search bar at the top of the page.




# DPO- Bhoomika

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig, get_peft_model

# Load your DPO data
import pandas as pd
final_dpo_df = pd.read_csv('final_dpo_df.csv')

# Check your column names first
print(final_dpo_df.columns.tolist())
final_dpo_df.head(10)

ModuleNotFoundError: No module named 'torch'

In [ ]:
!apt install trl

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package trl


In [ ]:
df = pd.read_csv('final_dpo_df.csv')

# Clean up - drop the extra index columns
df = df.drop(columns=['Unnamed: 0', ''], errors='ignore')
df = df.drop(columns=[c for c in df.columns if 'Unnamed' in c], errors='ignore')

print(f"Total samples: {len(df)}")
print(f"Ambiguous: {(df['is_ambiguous'] == 'True').sum()}")
print(f"Unambiguous: {(df['is_ambiguous'] == 'False').sum()}")


Total samples: 10036
Ambiguous: 0
Unambiguous: 0


In [ ]:
# Check the actual dtype and values
print(f"is_ambiguous dtype: {df['is_ambiguous'].dtype}")
print(f"Unique values: {df['is_ambiguous'].unique()}")

# Correct count
print(f"\nAmbiguous: {df['is_ambiguous'].sum()}")
print(f"Unambiguous: {(~df['is_ambiguous']).sum()}")

# Or if it's actually stored as string "True"/"False":
# print(f"Ambiguous: {(df['is_ambiguous'] == True).sum()}")

is_ambiguous dtype: bool
Unique values: [ True False]

Ambiguous: 4747
Unambiguous: 5289


In [ ]:
# Check the full positive and negative response format
print("=== AMBIGUOUS EXAMPLE ===")
amb_row = df[df['is_ambiguous'] == True].iloc[0]
print(f"Question: {amb_row['question']}")
print(f"Action: {amb_row['action']}")
print(f"Facets: {amb_row['facets']}")
print(f"Reasoning: {amb_row['reasoning']}")
print(f"Positive: {amb_row['positive_response']}")
print(f"Negative: {amb_row['neg_response']}")

print("\n=== UNAMBIGUOUS EXAMPLE ===")
unamb_row = df[df['is_ambiguous'] == False].iloc[0]
print(f"Question: {unamb_row['question']}")
print(f"Action: {unamb_row['action']}")
print(f"Facets: {unamb_row['facets']}")
print(f"Reasoning: {unamb_row['reasoning']}")
print(f"Positive: {unamb_row['positive_response']}")
print(f"Negative: {unamb_row['neg_response']}")

=== AMBIGUOUS EXAMPLE ===
Question: When did the simpsons first air on television?
Action: Clarify
Facets: ['Type of Show', 'Platform']
Reasoning: The question is ambiguous because it does not specify whether 'The Simpsons' refers to the TV series or another type of show, and it does not clarify if the platform (e.g., network, streaming service) is relevant. Without this information, we cannot determine the exact date of its first broadcast.
Positive: Are you asking about the original TV series or a movie, and on which platform (e.g., network, streaming service) did it first air?
Negative: The Simpsons first aired on December 17, 1989.

=== UNAMBIGUOUS EXAMPLE ===
Question: Who played george washington in the john adams series?
Action: Answer
Facets: []
Reasoning: The question is not ambiguous because it specifies the exact role and series, providing enough detail to uniquely identify the actor who portrayed George Washington in the John Adams miniseries.
Positive: ['David Morse']
Nega

In [ ]:
def format_chosen(row):
    """Format the full structured output as 'chosen' response"""
    return f"""Action: {row['action']}
Reasoning: {row['reasoning']}
Facets: {row['facets']}
Response: {row['positive_response']}"""

def format_rejected(row):
    """Format rejected with wrong action"""
    # Flip the action for rejected
    wrong_action = "Answer" if row['action'] == "Clarify" else "Clarify"
    wrong_facets = "[]" if row['action'] == "Clarify" else "['Unnecessary Clarification']"

    return f"""Action: {wrong_action}
Reasoning: Incorrect reasoning that led to wrong action.
Facets: {wrong_facets}
Response: {row['neg_response']}"""

def format_prompt(question):
    """Format the input prompt"""
    return f"""You are an intelligent assistant that determines whether a question is ambiguous.

Question: {question}

Analyze this question and respond with:
- Action: "Clarify" if ambiguous, "Answer" if not
- Reasoning: Brief explanation
- Facets: List of ambiguous aspects (if any)
- Response: Either a clarifying question or the direct answer"""

# Test on one example
test_row = df[df['is_ambiguous'] == True].iloc[0]
print("=== PROMPT ===")
print(format_prompt(test_row['question']))
print("\n=== CHOSEN ===")
print(format_chosen(test_row))
print("\n=== REJECTED ===")
print(format_rejected(test_row))

=== PROMPT ===
You are an intelligent assistant that determines whether a question is ambiguous.

Question: When did the simpsons first air on television?

Analyze this question and respond with:
- Action: "Clarify" if ambiguous, "Answer" if not
- Reasoning: Brief explanation
- Facets: List of ambiguous aspects (if any)
- Response: Either a clarifying question or the direct answer

=== CHOSEN ===
Action: Clarify
Reasoning: The question is ambiguous because it does not specify whether 'The Simpsons' refers to the TV series or another type of show, and it does not clarify if the platform (e.g., network, streaming service) is relevant. Without this information, we cannot determine the exact date of its first broadcast.
Facets: ['Type of Show', 'Platform']
Response: Are you asking about the original TV series or a movie, and on which platform (e.g., network, streaming service) did it first air?

=== REJECTED ===
Action: Answer
Reasoning: Incorrect reasoning that led to wrong action.
Facets

In [ ]:
# Build the DPO dataset
dpo_data = []

for _, row in df.iterrows():
    dpo_data.append({
        'prompt': format_prompt(row['question']),
        'chosen': format_chosen(row),
        'rejected': format_rejected(row)
    })

# Convert to HuggingFace Dataset
from datasets import Dataset
dpo_dataset = Dataset.from_list(dpo_data)

print(f"DPO Dataset: {len(dpo_dataset)} samples")
print(f"Columns: {dpo_dataset.column_names}")

# Train/eval split
dpo_dataset = dpo_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = dpo_dataset['train']
eval_dataset = dpo_dataset['test']

print(f"Train: {len(train_dataset)}")
print(f"Eval: {len(eval_dataset)}")

# Print full sample
print("=== FULL SAMPLE FROM TRAIN ===\n")
print("PROMPT:")
print(train_dataset[0]['prompt'])
print("\n" + "="*50 + "\n")
print("CHOSEN:")
print(train_dataset[0]['chosen'])
print("\n" + "="*50 + "\n")
print("REJECTED:")
print(train_dataset[0]['rejected'])

DPO Dataset: 10036 samples
Columns: ['prompt', 'chosen', 'rejected']
Train: 9534
Eval: 502
=== FULL SAMPLE FROM TRAIN ===

PROMPT:
You are an intelligent assistant that determines whether a question is ambiguous.

Question: What is the name of the first ewok leia meets?

Analyze this question and respond with:
- Action: "Clarify" if ambiguous, "Answer" if not
- Reasoning: Brief explanation
- Facets: List of ambiguous aspects (if any)
- Response: Either a clarifying question or the direct answer


CHOSEN:
Action: Answer
Reasoning: The question is not ambiguous because it specifies the exact scenario of Leia meeting an Ewok, and there is only one first meeting described in the original Star Wars canon. This uniquely identifies a single character.
Facets: []
Response: ['Wicket Widget Warrick', 'Wicket W. Warrick']


REJECTED:
Action: Clarify
Reasoning: Incorrect reasoning that led to wrong action.
Facets: ['Unnecessary Clarification']
Response: Could you confirm if you're referring to the

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print(f"Tokenizer loaded: {tokenizer.name_or_path}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Pad token: {tokenizer.pad_token}")

Tokenizer loaded: Qwen/Qwen2.5-7B-Instruct
Vocab size: 151643
Pad token: <|im_end|>


In [ ]:
!nvidia-smi

Tue Nov 25 20:27:14 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:00:05.0 Off |                    0 |
| N/A   26C    P0             82W /  500W |   36893MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# Restarting session after downloading everything!

# LoRa and DPO config

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = DPOConfig(
    output_dir="./dpo_output",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=20,
    save_steps=200,
    eval_steps=200,
    eval_strategy="steps",
    bf16=True,
    max_length=512,
    max_prompt_length=384,
    beta=0.1,
    remove_unused_columns=False,
    report_to="none",
)

print("LoRA and DPO configs ready")

LoRA and DPO configs ready


# Training Iteration 1

In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig

print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

torch: 2.7.1+cu118
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [ ]:
 model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print(f"Tokenizer loaded: {model_name}")

Tokenizer loaded: Qwen/Qwen2.5-7B-Instruct


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model loaded: {model.config._name_or_path}")

Loading checkpoint shards: 100%|██████████| 4/4 [00:50<00:00, 12.74s/it]

Model loaded: Qwen/Qwen2.5-7B-Instruct


In [ ]:
trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print(f"Trainer created")
print(f"Total steps: {len(train_dataset) // (8 * 2)}")

/opt/conda/lib/python3.11/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Tokenizing eval dataset: 100%|██████████| 502/502 [00:00<00:00, 1338.59 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.


Trainer created
Total steps: 595


In [ ]:
!conda install -y gxx_linux-64 gcc_linux-64 -c conda-forge

Solving environment: done


==> WARNING: A newer version of conda exists. <==
  current version: 23.9.0
  latest version: 25.11.0

Please update conda by running

    $ conda update -n base -c conda-forge conda

Or to minimize the number of packages updated during conda update use

     conda install conda=25.11.0



# All requested packages already installed.



In [ ]:
import os
os.environ['CC'] = '/opt/conda/bin/x86_64-conda-linux-gnu-gcc'
os.environ['CXX'] = '/opt/conda/bin/x86_64-conda-linux-gnu-g++'

In [ ]:
print("Starting DPO training...")
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting DPO training...


/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
200,0.000100,0.000129,1.201322,-8.145965,1.000000,9.347287,-144.585495,-218.983871,-0.708353,-0.726422
400,0.000100,0.000064,1.173926,-8.870489,1.000000,10.044416,-144.859467,-226.229126,-0.759574,-0.746532


/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


TrainOutput(global_step=596, training_loss=0.04845607046357468, metrics={'train_runtime': 1779.245, 'train_samples_per_second': 5.358, 'train_steps_per_second': 0.335, 'total_flos': 0.0, 'train_loss': 0.04845607046357468, 'epoch': 1.0})

In [ ]:
trainer.save_model("./dpo_final")
tokenizer.save_pretrained("./dpo_final")
print("Model saved to ./dpo_final")

Model saved to ./dpo_final


# Test the model


In [ ]:
from peft import PeftModel

# Load the trained model for inference
model.eval()

def test_question(question):
    prompt = format_prompt(question)

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

# Test on ambiguous question
print("=== AMBIGUOUS QUESTION ===")
print("Q: When did the simpsons first air on television?")
print(test_question("When did the simpsons first air on television?"))

print("\n" + "="*50 + "\n")

# Test on unambiguous question
print("=== UNAMBIGUOUS QUESTION ===")
print("Q: Who played george washington in the john adams series?")
print(test_question("Who played george washington in the john adams series?"))

=== AMBIGUOUS QUESTION ===
Q: When did the simpsons first air on television?
Action: Answer

Reasoning: The question is not ambiguous as it clearly asks for a specific date or time period when the TV show "The Simpsons" first aired.

Facets: None

Response: The Simpsons first aired on December 17, 1989, on Fox.


=== UNAMBIGUOUS QUESTION ===
Q: Who played george washington in the john adams series?
Action: Clarify
Reasoning: The question is ambiguous because it does not specify which adaptation of John Adams' life or presidency is being referred to. There have been multiple productions about John Adams and his era.
Facets: 
- Which specific production or series of "John Adams" is being asked about (e.g., the 2008 HBO miniseries, a stage play, etc.)

Response: Could you please specify which adaptation or series of "John Adams" you are referring to?


### Ambiguous question → Model says "Answer" (should be "Clarify")
### Unambiguous question → Model says "Clarify" (should be "Answer")

#This suggests the DPO training may not have properly loaded on top of the base model, or the LoRA weights aren't being #applied during inference.
#Let's check and fix:

In [ ]:
# Check if LoRA is active
print(f"Model type: {type(model)}")
print(f"Is PEFT model: {hasattr(model, 'peft_config')}")

# List adapter info
if hasattr(model, 'peft_config'):
    print(f"Adapters: {model.peft_config}")

Model type: <class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
Is PEFT model: True
Adapters: {'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.0', base_model_name_or_path='Qwen/Qwen2.5-7B-Instruct', revision=None, inference_mode=False, r=16, target_modules={'v_proj', 'down_proj', 'gate_proj', 'k_proj', 'up_proj', 'o_proj', 'q_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_paramete

In [ ]:
model.eval()

def test_question(question):
    prompt = format_prompt(question)

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

# Test again
print("=== AMBIGUOUS ===")
print("Q: What is the legal age of marriage in usa?")
print(test_question("What is the legal age of marriage in usa?"))

print("\n" + "="*50 + "\n")

print("=== UNAMBIGUOUS ===")
print("Q: When did the frozen ride open at epcot?")
print(test_question("When did the frozen ride open at epcot?"))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== AMBIGUOUS ===
Q: What is the legal age of marriage in usa?
Action: Clarify
Reasoning: The question is ambiguous because it does not specify which state within the USA is being referred to, as marriage laws can vary from one state to another.
Facets: 
- State specificity: The question does not mention a particular state.

Response: In which U.S. state would you like to know the legal age of marriage?


=== UNAMBIGUOUS ===
Q: When did the frozen ride open at epcot?
Action: Clarify
Reasoning: The question is ambiguous because it does not specify which "frozen ride" is being referred to. EPCOT has multiple rides, and without additional context, it's unclear which one is meant.
Facets: 
- Specific frozen ride not mentioned

Response: Could you please specify which frozen ride you're asking about, such as "Frozen Ever After" or another attraction?


In [ ]:
# Test clearly unambiguous questions
test_cases = [
    "What is the capital of France?",
    "Who wrote Romeo and Juliet?",
    "What year did World War 2 end?",
    "Who is the first president of the United States?",
]

for q in test_cases:
    print(f"Q: {q}")
    result = test_question(q)
    action = result.split('\n')[0] if result else "N/A"
    print(f"{action}\n")

Q: What is the capital of France?
Action: Answer

Q: Who wrote Romeo and Juliet?
Action: Answer

Q: What year did World War 2 end?
Action: Answer

Q: Who is the first president of the United States?
Action: Answer



In [ ]:
# Test cases from your dataset
test_cases = [
    ("When did the simpsons first air on television?", "Ambiguous"),
    ("Who played george washington in the john adams series?", "Unambiguous"),
    ("When was the last time uga won a national championship?", "Ambiguous"),
    ("What is the name of the first ewok leia meets?", "Unambiguous"),
    ("Who sing play that funky music white boy?", "Ambiguous"),
]

for q, expected in test_cases:
    print(f"Q: {q}")
    print(f"Expected: {expected}")
    result = test_question(q)
    print(result[:200] + "..." if len(result) > 200 else result)
    print("-"*50)

Q: When did the simpsons first air on television?
Expected: Ambiguous
Action: Answer

Reasoning: The question is not ambiguous as it clearly asks for a specific date or time period when the TV show "The Simpsons" first aired.

Facets: None

Response: The Simpsons first ...
--------------------------------------------------
Q: Who played george washington in the john adams series?
Expected: Unambiguous
Action: Clarify
Reasoning: The question is ambiguous because it does not specify which version of the John Adams series is being referred to, as there could be multiple adaptations.
Facets: 
- Version...
--------------------------------------------------
Q: When was the last time uga won a national championship?
Expected: Ambiguous
Action: Clarify
Reasoning: The question is ambiguous because "uga" could refer to multiple entities, such as UGA (University of Georgia), UGA (University of Guadalajara), or even a nickname for a pers...
--------------------------------------------------
Q: Wha

### The model is over-clarifying - it finds ambiguity even in questions labeled as unambiguous in your dataset.

### The issue: Your DPO training taught the model the format well, but the distinction between when to clarify vs answer isn't matching your dataset's labels exactly.

In [ ]:
def format_rejected_v2(row):
    """Format rejected with more realistic wrong reasoning"""
    if row['action'] == "Clarify":
        # Wrong: Answering when should clarify
        return f"""Action: Answer
Reasoning: The question is clear and specific enough to answer directly without needing clarification.
Facets: []
Response: {row['neg_response']}"""
    else:
        # Wrong: Clarifying when should answer
        return f"""Action: Clarify
Reasoning: The question might have multiple interpretations and could benefit from clarification.
Facets: ['Possible Ambiguity']
Response: {row['neg_response']}"""

# Test it
test_row = df[df['is_ambiguous'] == True].iloc[0]
print("=== CHOSEN ===")
print(format_chosen(test_row))
print("\n=== REJECTED V2 ===")
print(format_rejected_v2(test_row))

# Step 2: Rebuild dataset with better rejected responses

In [ ]:
dpo_data_v2 = []
for _, row in df.iterrows():
    dpo_data_v2.append({
        'prompt': format_prompt(row['question']),
        'chosen': format_chosen(row),
        'rejected': format_rejected_v2(row)
    })

dpo_dataset_v2 = Dataset.from_list(dpo_data_v2)
dpo_dataset_v2 = dpo_dataset_v2.train_test_split(test_size=0.05, seed=42)
train_dataset_v2 = dpo_dataset_v2['train']
eval_dataset_v2 = dpo_dataset_v2['test']

print(f"Train: {len(train_dataset_v2)}")
print(f"Eval: {len(eval_dataset_v2)}")

Train: 9534
Eval: 502


In [ ]:
# Clear memory
del model
del trainer
import gc
gc.collect()
torch.cuda.empty_cache()

# Load fresh model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

print("Fresh model loaded")

Loading checkpoint shards: 100%|██████████| 4/4 [01:02<00:00, 15.58s/it]

Fresh model loaded


In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = DPOConfig(
    output_dir="./dpo_output_v2",
    num_train_epochs=2,              # More epochs
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=20,
    save_steps=300,
    eval_steps=300,
    eval_strategy="steps",
    bf16=True,
    max_length=512,
    max_prompt_length=384,
    beta=0.3,                        # Stronger preference signal
    remove_unused_columns=False,
    report_to="none",
)

print("Configs ready - beta=0.3, epochs=2")

Configs ready - beta=0.3, epochs=2


In [ ]:
trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_v2,
    eval_dataset=eval_dataset_v2,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print(f"Trainer created")
print(f"Total steps: {len(train_dataset_v2) * 2 // (8 * 2)}")  # 2 epochs
print("Starting training...")
trainer.train()

Tokenizing eval dataset: 100%|██████████| 502/502 [00:00<00:00, 1352.20 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Trainer created
Total steps: 1191
Starting training...


/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
300,0.000000,0.000050,4.245902,-6.960986,1.000000,11.206887,-142.445709,-129.441437,-0.120031,-0.160683
600,0.000000,0.000018,4.377250,-7.899291,1.000000,12.276541,-142.007889,-132.569107,-0.144251,-0.175105
900,0.000000,0.000013,4.423460,-8.246720,1.000000,12.670181,-141.853851,-133.727219,-0.151568,-0.179715


/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


TrainOutput(global_step=1192, training_loss=0.029341716619181158, metrics={'train_runtime': 3084.1586, 'train_samples_per_second': 6.183, 'train_steps_per_second': 0.386, 'total_flos': 0.0, 'train_loss': 0.029341716619181158, 'epoch': 2.0})

In [ ]:
# Quick test on the trained model (still in memory)
model.eval()

test_prompt = """Analyze the following question for ambiguity and provide structured facets.

Question: When did the movie come out?

Provide your response in this format:
Action: [CLARIFY or ANSWER]
Reasoning: [Your reasoning]
Facets: [If CLARIFY, list the facets]
Response: [Clarifying question or direct answer]"""

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Analyze the following question for ambiguity and provide structured facets.

Question: When did the movie come out?

Provide your response in this format:
Action: [CLARIFY or ANSWER]
Reasoning: [Your reasoning]
Facets: [If CLARIFY, list the facets]
Response: [Clarifying question or direct answer] Action: CLARIFY
Reasoning: The question "When did the movie come out?" is ambiguous because it does not specify which movie is being referred to.
Facets: 
1. Genre of the movie
2. Release date
3. Name of the movie
4. Country of origin

Response: Could you please specify which movie you are referring to? Additionally, would you like to know the release date in a specific country or globally? To narrow down the information, could you also provide the genre or name of the movie? Your clarification will help me provide you with the accurate information you need. 

Note: The response provided is more detailed than the original request, but it gives a clear structure as per the guidelines. In practi

# Redoing DPO using SFT LoRa model

In [ ]:
import pandas as pd

df = pd.read_csv("final_dpo_df.csv")
print(f"Loaded {len(df)} rows")

Loaded 10036 rows


In [ ]:
!apt-get install -y gcc g++

E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
SFT_ADAPTER = "bhavanasanghi9/clarifier-sft-lora"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

print("Loading SFT adapter...")
model = PeftModel.from_pretrained(base_model, SFT_ADAPTER)
model.config.use_cache = False

print("✅ Model loaded!")

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model...


Loading checkpoint shards: 100%|██████████| 4/4 [00:50<00:00, 12.66s/it]


Loading SFT adapter...
✅ Model loaded!


In [ ]:
!pip uninstall bitsandbytes -y

Found existing installation: bitsandbytes 0.48.2
Uninstalling bitsandbytes-0.48.2:
  Successfully uninstalled bitsandbytes-0.48.2


In [ ]:
import pandas as pd
import ast

df = pd.read_csv("final_dpo_df.csv")
print(f"✅ Loaded {len(df)} rows")
print(df.columns.tolist())

✅ Loaded 10036 rows
['Unnamed: 0.1', 'Unnamed: 0', 'question', 'is_ambiguous', 'facets', 'reasoning', 'positive_response', 'action', 'neg_response']


In [ ]:
def clean_facets(f):
    if isinstance(f, list):
        return f
    if isinstance(f, str):
        try:
            v = ast.literal_eval(f)
            return v if isinstance(v, list) else []
        except:
            return []
    return []

def clean_response(r):
    if r is None:
        return ""
    if isinstance(r, list):
        return r[0] if r else ""
    if isinstance(r, str) and r.startswith("[") and r.endswith("]"):
        try:
            lst = ast.literal_eval(r)
            return lst[0] if isinstance(lst, list) and lst else r
        except:
            pass
    return str(r).strip()

SYSTEM_PROMPT = (
    "You are a question understanding agent trained on AmbigQA-style ambiguity. "
    "For each user question:\n"
    "1) Decide if it is ambiguous (i.e., multiple plausible interpretations/answers).\n"
    "2) Explain your reasoning.\n"
    "3) List facets if ambiguous.\n"
    "4) Ask a clarifying question OR give a direct answer.\n\n"
    "Format:\n"
    "Action: Clarify|Answer\n"
    "Reasoning: ...\n"
    "Facets: [...]\n"
    "Response: ...\n"
)

def format_dpo_row(row):
    question = row["question"]
    action = row["action"]
    reasoning = row["reasoning"]
    facets = clean_facets(row["facets"])
    pos_response = clean_response(row["positive_response"])
    neg_response = clean_response(row["neg_response"])

    facets_str = "[" + ", ".join(facets) + "]" if facets else "[]"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    chosen = f"Action: {action}\nReasoning: {reasoning}\nFacets: {facets_str}\nResponse: {pos_response}"

    if action == "Clarify":
        rejected = f"Action: Answer\nReasoning: {reasoning}\nFacets: []\nResponse: {neg_response}"
    else:
        rejected = f"Action: Clarify\nReasoning: {reasoning}\nFacets: {facets_str}\nResponse: {neg_response}"

    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

print("✅ Helper functions defined")

✅ Helper functions defined


In [ ]:
from datasets import Dataset

print("Formatting dataset...")
dpo_data = [format_dpo_row(row) for _, row in df.iterrows()]
dpo_df = pd.DataFrame(dpo_data)

# Remove empty rows
dpo_df = dpo_df[(dpo_df["prompt"].str.len() > 0) & (dpo_df["chosen"].str.len() > 0) & (dpo_df["rejected"].str.len() > 0)]
print(f"Formatted {len(dpo_df)} examples")

# Shuffle and split
dpo_df = dpo_df.sample(frac=1, random_state=42).reset_index(drop=True)
eval_size = 500

train_dataset = Dataset.from_pandas(dpo_df.iloc[eval_size:][["prompt", "chosen", "rejected"]])
eval_dataset = Dataset.from_pandas(dpo_df.iloc[:eval_size][["prompt", "chosen", "rejected"]])

print(f"✅ Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

Formatting dataset...
Formatted 10036 examples
✅ Train: 9536, Eval: 500


In [ ]:
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = DPOConfig(
    output_dir="./dpo_model",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    beta=0.3,
    max_length=1024,
    max_prompt_length=512,
    logging_steps=50,
    save_steps=300,
    eval_steps=300,
    eval_strategy="steps",
    bf16=True,
    warmup_ratio=0.1,
    optim="adamw_torch",
    remove_unused_columns=False,
    report_to="none",
)

print("✅ Training config ready")

✅ Training config ready


In [ ]:
trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print(f"Total training steps: ~{len(train_dataset) * 2 // (2 * 8)}")
print("Starting training...")
trainer.train()

/opt/conda/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Tokenizing eval dataset: 100%|██████████| 500/500 [00:00<00:00, 1315.03 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Total training steps: ~1192
Starting training...


/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
300,0.000000,0.000002,-0.558093,-21.357016,1.000000,20.798922,-15.285074,-133.221786,-1.230859,-0.728339
600,0.000000,0.000001,-0.565488,-21.738817,1.000000,21.173330,-15.309725,-134.494446,-1.235903,-0.745941
900,0.000000,0.000001,-0.561483,-21.935246,1.000000,21.373764,-15.296377,-135.149216,-1.237502,-0.753580


/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


TrainOutput(global_step=1192, training_loss=0.011987933895815997, metrics={'train_runtime': 4054.181, 'train_samples_per_second': 4.704, 'train_steps_per_second': 0.294, 'total_flos': 0.0, 'train_loss': 0.011987933895815997, 'epoch': 2.0})

In [ ]:
trainer.save_model("./dpo_model_final")
tokenizer.save_pretrained("./dpo_model_final")
print("✅ Model saved to ./dpo_model_final")

✅ Model saved to ./dpo_model_final


In [ ]:
model.eval()

test_questions = [
    "When did the movie come out?",
    "Who won the championship?",
    "What is the capital of France?",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.3,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract assistant part
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()

    print(f"\n{'='*50}")
    print(f"Q: {q}")
    print(f"{'='*50}")
    print(response[:500])


Q: When did the movie come out?
Action: Clarify
Reasoning: The question is ambiguous because it does not specify which movie is being referred to or in which geographical region the release date is asked about. Without knowing the movie title and the specific region, there could be multiple correct answers.
Facets: [Movie Title, Geographical Region]
Response: Could you specify the title of the movie and the geographical region you're interested in?

Q: Who won the championship?
Action: Clarify
Reasoning: The question is ambiguous because it does not specify the type of championship (e.g., sports, academic, business) or the year of the event. Without this information, it's unclear which specific championship is being referred to, and thus who won it.
Facets: [Type of Championship, Year]
Response: Could you specify the type of championship and the year you're asking about?

Q: What is the capital of France?
Action: Answer
Reasoning: The question is not ambiguous because it clearly asks 

In [ ]:
!pip install ipywidgets

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 2.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 15.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 43.2 MB/s eta 0:00:00:00:01


In [ ]:
from huggingface_hub import login

login(token="hf_REDACTED_TOKEN")

In [ ]:
from huggingface_hub import create_repo, upload_folder

create_repo(repo_id="itsbhoomika/clarifier-dpo-lora", exist_ok=True)

upload_folder(
    folder_path="./dpo_model_final",
    repo_id="itsbhoomika/clarifier-dpo-lora",
    repo_type="model",
)

print("✅ Uploaded!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 2):   7%|▋         | 12.0MB /  173MB, 1.04MB/s  
Processing Files (1 / 2):  15%|█▌        | 26.4MB /  173MB, 18.7MB/s  
Processing Files (1 / 3):  35%|███▍      | 60.1MB /  173MB, 48.7MB/s  
Processing Files (1 / 3):  76%|███████▌  |  131MB /  173MB, 99.4MB/s  
Processing Files (1 / 3):  95%|█████████▍|  164MB /  173MB,  109MB/s  
Processing Files (2 / 3):  99%|█████████▉|  172MB /  173MB,  100MB/s  
Processing Files (2 / 3): 100%|█████████▉|  173MB /  173MB, 80.6MB/s  
Processing Files (3 / 3): 100%|██████████|  173MB /  173MB, 73.4MB/s  
Processing Files (3 / 3): 100%|██████████|  173MB /  173MB, 62.1MB/s  
New Data Upload: 100%|██████████|  161MB /  161MB, 62.1MB/s  


✅ Uploaded!


# Evaluation

In [ ]:
import pandas as pd
import torch
from tqdm import tqdm

# Load evaluation data
eval_df = pd.read_csv("final_dpo_df.csv")
eval_df = eval_df.sample(n=200, random_state=42).reset_index(drop=True)  # Sample for evaluation
print(f"Evaluating on {len(eval_df)} samples")

Evaluating on 200 samples


In [ ]:
from tqdm import tqdm

def generate_prediction(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.3,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()
    return response

# Generate predictions
print("Generating predictions...")
predictions = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    pred = generate_prediction(row["question"])
    predictions.append(pred)

eval_df["prediction"] = predictions
print("✅ Predictions generated")

Generating predictions...


100%|██████████| 200/200 [12:34<00:00,  3.77s/it]

✅ Predictions generated


In [ ]:
def parse_prediction(pred_text):
    """Extract Action, Reasoning, Facets, Response from model output"""
    result = {
        "pred_action": None,
        "pred_reasoning": "",
        "pred_facets": [],
        "pred_response": ""
    }

    lines = pred_text.strip().split("\n")

    for line in lines:
        line_lower = line.lower().strip()
        if line_lower.startswith("action:"):
            val = line.split(":", 1)[1].strip()
            if val in ["Clarify", "Answer"]:
                result["pred_action"] = val
            elif "clarify" in val.lower():
                result["pred_action"] = "Clarify"
            elif "answer" in val.lower():
                result["pred_action"] = "Answer"
        elif line_lower.startswith("reasoning:"):
            result["pred_reasoning"] = line.split(":", 1)[1].strip()
        elif line_lower.startswith("facets:"):
            facet_str = line.split(":", 1)[1].strip()
            try:
                result["pred_facets"] = ast.literal_eval(facet_str)
            except:
                # Parse [X, Y, Z] manually
                facet_str = facet_str.strip("[]")
                if facet_str:
                    result["pred_facets"] = [f.strip() for f in facet_str.split(",")]
        elif line_lower.startswith("response:"):
            result["pred_response"] = line.split(":", 1)[1].strip()

    return result

# Parse all predictions
parsed = [parse_prediction(p) for p in eval_df["prediction"]]
eval_df["pred_action"] = [p["pred_action"] for p in parsed]
eval_df["pred_reasoning"] = [p["pred_reasoning"] for p in parsed]
eval_df["pred_facets"] = [p["pred_facets"] for p in parsed]
eval_df["pred_response"] = [p["pred_response"] for p in parsed]

print("✅ Predictions parsed")
print(f"Sample parsed: {parsed[0]}")

✅ Predictions parsed
Sample parsed: {'pred_action': 'Answer', 'pred_reasoning': 'The question is not ambiguous because it specifies the exact anatomical structure (secondary ossification center in a long bone) being asked about. This description provides enough detail to uniquely determine the location within the bone.', 'pred_facets': [], 'pred_response': 'epiphyses'}


In [ ]:
# Metric 1: Action Accuracy
action_correct = (eval_df["pred_action"] == eval_df["action"]).sum()
action_accuracy = action_correct / len(eval_df)
print(f"1. Action Accuracy: {action_accuracy:.3f} ({action_correct}/{len(eval_df)})")

# Metric 2: Clarify Precision/Recall
clarify_gold = eval_df["action"] == "Clarify"
clarify_pred = eval_df["pred_action"] == "Clarify"

true_positives = (clarify_gold & clarify_pred).sum()
false_positives = (~clarify_gold & clarify_pred).sum()
false_negatives = (clarify_gold & ~clarify_pred).sum()

clarify_precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
clarify_recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
clarify_f1 = 2 * clarify_precision * clarify_recall / (clarify_precision + clarify_recall) if (clarify_precision + clarify_recall) > 0 else 0

print(f"\n2. Clarify Detection:")
print(f"   Precision: {clarify_precision:.3f}")
print(f"   Recall: {clarify_recall:.3f}")
print(f"   F1: {clarify_f1:.3f}")

# Metric 3: Facet Generation (for Clarify cases)
clarify_rows = eval_df[eval_df["action"] == "Clarify"]
facet_generated = (clarify_rows["pred_facets"].apply(len) > 0).sum()
facet_rate = facet_generated / len(clarify_rows) if len(clarify_rows) > 0 else 0
print(f"\n3. Facet Generation Rate (when Clarify): {facet_rate:.3f} ({facet_generated}/{len(clarify_rows)})")

# Metric 4: Average number of facets when clarifying
avg_facets = clarify_rows["pred_facets"].apply(len).mean()
print(f"   Average facets per Clarify: {avg_facets:.2f}")

# Metric 5: Answer cases - response provided
answer_rows = eval_df[eval_df["action"] == "Answer"]
answer_pred_correct = eval_df[(eval_df["action"] == "Answer") & (eval_df["pred_action"] == "Answer")]
answer_has_response = (answer_pred_correct["pred_response"].str.len() > 0).sum()
answer_response_rate = answer_has_response / len(answer_pred_correct) if len(answer_pred_correct) > 0 else 0
print(f"\n4. Answer Response Rate: {answer_response_rate:.3f} ({answer_has_response}/{len(answer_pred_correct)})")

print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print(f"Action Accuracy:     {action_accuracy:.3f}")
print(f"Clarify F1:          {clarify_f1:.3f}")
print(f"Facet Generation:    {facet_rate:.3f}")
print(f"Answer Response:     {answer_response_rate:.3f}")

1. Action Accuracy: 0.595 (119/200)

2. Clarify Detection:
   Precision: 0.941
   Recall: 0.168
   F1: 0.286

3. Facet Generation Rate (when Clarify): 0.189 (18/95)
   Average facets per Clarify: 0.34

4. Answer Response Rate: 1.000 (103/103)

SUMMARY
Action Accuracy:     0.595
Clarify F1:          0.286
Facet Generation:    0.189
Answer Response:     1.000


In [ ]:
# Check prediction distribution vs gold distribution
print("Gold distribution:")
print(eval_df["action"].value_counts())

print("\nPredicted distribution:")
print(eval_df["pred_action"].value_counts())

# Check confusion matrix
print("\nConfusion Matrix:")
print(pd.crosstab(eval_df["action"], eval_df["pred_action"], rownames=["Gold"], colnames=["Pred"]))

# Look at some false negatives (should Clarify but predicted Answer)
false_negatives = eval_df[(eval_df["action"] == "Clarify") & (eval_df["pred_action"] == "Answer")]
print(f"\nFalse Negatives (missed Clarify): {len(false_negatives)}")
print("\nExamples of missed Clarify:")
for i, row in false_negatives.head(3).iterrows():
    print(f"\nQ: {row['question']}")
    print(f"Gold: Clarify | Pred: Answer")
    print(f"Model response: {row['pred_response'][:100]}...")

Gold distribution:
action
Answer     105
Clarify     95
Name: count, dtype: int64

Predicted distribution:
pred_action
Answer     182
Clarify     17
Name: count, dtype: int64

Confusion Matrix:
Pred     Answer  Clarify
Gold                    
Answer      103        1
Clarify      79       16

False Negatives (missed Clarify): 79

Examples of missed Clarify:

Q: Who remade mad world by tears for fears?
Gold: Clarify | Pred: Answer
Model response: Tears for Fears...

Q: How many house of cards seasons are there?
Gold: Clarify | Pred: Answer
Model response: 5...

Q: When was five nights at freddy's 2 made?
Gold: Clarify | Pred: Answer
Model response: September 25, 2014...


# model is over-answering

In [ ]:
# Test with lower temperature - might help
test_q = "Who remade mad world by tears for fears?"
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_q},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        temperature=0.7,  # Try higher temperature
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response.split("assistant")[-1].strip())

Action: Answer
Reasoning: The question is not ambiguous because it specifies the original song and artist, and asks about a remake. The details provided uniquely identify the question without any missing information.
Facets: []
Response: Tears for Fears


In [ ]:
# Load the original SFT model (without DPO)
from transformers import AutoModelForCausalLM
from peft import PeftModel

print("Loading SFT model (without DPO)...")
sft_base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
sft_model = PeftModel.from_pretrained(sft_base, "bhavanasanghi9/clarifier-sft-lora")
sft_model.eval()

# Test same questions
test_questions = [
    "Who remade mad world by tears for fears?",
    "How many house of cards seasons are there?",
    "When did the movie come out?",
]

print("\n" + "="*50)
print("SFT MODEL (before DPO)")
print("="*50)

for q in test_questions:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(sft_model.device)

    with torch.no_grad():
        outputs = sft_model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.3,
            do_sample=True,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()

    print(f"\nQ: {q}")
    print(response[:300])
    print("-"*40)

Loading SFT model (without DPO)...


Loading checkpoint shards: 100%|██████████| 4/4 [00:50<00:00, 12.67s/it]



SFT MODEL (before DPO)

Q: Who remade mad world by tears for fears?
Action: Answer
Reasoning: The question is not ambiguous because it specifies the original song and asks about its remake, providing enough detail to uniquely determine the answer. There is no missing information that would cause ambiguity.
Facets: []
Response: Tears for Fears
----------------------------------------

Q: How many house of cards seasons are there?
Action: Answer
Reasoning: The question is not ambiguous because it specifically asks about the number of seasons for the TV show "House of Cards." This question has a single, well-known answer based on the existing content.
Facets: []
Response: 5
----------------------------------------

Q: When did the movie come out?
Action: Clarify
Reasoning: The question is ambiguous because it does not specify which geographical region, language version, or release type (e.g., theatrical, digital) is being referred to. Different regions and versions of a movie can have di

 ## The root cause is in your training data. Questions like "Who remade mad world by tears for fears?" are labeled as "Clarify" in your gold data, but the model learned that specific entity questions should be answered directly.
Two paths forward:
Option A: Accept current behavior, adjust gold labels
The model's behavior is actually reasonable - "Who remade mad world" has a well-known answer (Gary Jules). The ambiguity in your gold data may be too strict.
Option B: Retrain with adjusted data
If you want more clarification, you need to ensure training examples like these are clearly labeled as Clarify with strong signal.
Let's check your gold data for these specific questions:                          

In [ ]:
# Check what the gold data says for these questions
check_questions = [
    "Who remade mad world by tears for fears?",
    "How many house of cards seasons are there?",
]

for q in check_questions:
    match = eval_df[eval_df["question"].str.lower() == q.lower()]
    if len(match) > 0:
        row = match.iloc[0]
        print(f"\nQ: {q}")
        print(f"Gold Action: {row['action']}")
        print(f"Gold Facets: {row['facets']}")
        print(f"Gold Reasoning: {row['reasoning'][:200]}...")


Q: Who remade mad world by tears for fears?
Gold Action: Clarify
Gold Facets: ['Year', 'Month']
Gold Reasoning: The question is ambiguous because it does not specify the year or month of the remake. Without this information, it is unclear which version of "Mad World" is being referred to, as multiple artists ma...

Q: How many house of cards seasons are there?
Gold Action: Clarify
Gold Facets: ['Year', 'Country of Origin']
Gold Reasoning: The question is ambiguous because it does not specify which version of "House of Cards" is being referred to. There could be different numbers of seasons depending on the year and country of origin, a...


In [ ]:
# Check DPO data - how many Clarify vs Answer in training?
full_df = pd.read_csv("final_dpo_df.csv")
print("Full DPO data distribution:")
print(full_df["action"].value_counts())

# Check if facets are actually different between chosen and rejected
print("\n\nSample chosen vs rejected for a Clarify case:")
clarify_sample = full_df[full_df["action"] == "Clarify"].iloc[0]
print(f"Question: {clarify_sample['question']}")
print(f"Action: {clarify_sample['action']}")
print(f"Positive Response: {clarify_sample['positive_response'][:150]}...")
print(f"Negative Response: {clarify_sample['neg_response'][:150]}...")

Full DPO data distribution:
action
Answer     5289
Clarify    4747
Name: count, dtype: int64


Sample chosen vs rejected for a Clarify case:
Question: When did the simpsons first air on television?
Action: Clarify
Positive Response: Are you asking about the original TV series or a movie, and on which platform (e.g., network, streaming service) did it first air?...
Negative Response: The Simpsons first aired on December 17, 1989....


In [ ]:
# Run same 200 samples through SFT model for comparison
print("Generating SFT predictions for comparison...")
sft_predictions = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["question"]},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(sft_model.device)

    with torch.no_grad():
        outputs = sft_model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.3,
            do_sample=True,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()
    sft_predictions.append(response)

eval_df["sft_prediction"] = sft_predictions

# Parse SFT predictions
sft_parsed = [parse_prediction(p) for p in sft_predictions]
eval_df["sft_action"] = [p["pred_action"] for p in sft_parsed]

# Compare
print("\n" + "="*50)
print("COMPARISON: SFT vs DPO")
print("="*50)

sft_accuracy = (eval_df["sft_action"] == eval_df["action"]).mean()
dpo_accuracy = (eval_df["pred_action"] == eval_df["action"]).mean()

print(f"SFT Action Accuracy: {sft_accuracy:.3f}")
print(f"DPO Action Accuracy: {dpo_accuracy:.3f}")

print("\nSFT distribution:")
print(eval_df["sft_action"].value_counts())

print("\nDPO distribution:")
print(eval_df["pred_action"].value_counts())

Generating SFT predictions for comparison...


100%|██████████| 200/200 [11:55<00:00,  3.58s/it]


COMPARISON: SFT vs DPO
SFT Action Accuracy: 0.590
DPO Action Accuracy: 0.595

SFT distribution:
sft_action
Answer     181
Clarify     18
Name: count, dtype: int64

DPO distribution:
pred_action
Answer     182
Clarify     17
Name: count, dtype: int64


In [ ]:
STRONGER_PROMPT = (
    "You are a question understanding agent trained on AmbigQA-style ambiguity. "
    "IMPORTANT: Many questions that seem simple are actually ambiguous. "
    "Consider: Could this question have multiple valid answers? Are there missing details "
    "(time period, location, version, entity)? If ANY ambiguity exists, you MUST clarify.\n\n"
    "For each user question:\n"
    "1) Decide if it is ambiguous (i.e., multiple plausible interpretations/answers).\n"
    "2) Explain your reasoning.\n"
    "3) List facets if ambiguous.\n"
    "4) Ask a clarifying question OR give a direct answer.\n\n"
    "Format:\n"
    "Action: Clarify|Answer\n"
    "Reasoning: ...\n"
    "Facets: [...]\n"
    "Response: ...\n"
)

test_questions = [
    "Who remade mad world by tears for fears?",
    "How many house of cards seasons are there?",
    "When did the movie come out?",
    "What is the capital of France?",
]

print("Testing with STRONGER prompt:\n")

for q in test_questions:
    messages = [
        {"role": "system", "content": STRONGER_PROMPT},
        {"role": "user", "content": q},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.3,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()

    print(f"Q: {q}")
    print(response[:350])
    print("-"*50)

Testing with STRONGER prompt:

Q: Who remade mad world by tears for fears?
Action: Answer
Reasoning: The question is not ambiguous because it specifies the original song and artist. These details uniquely identify the remake being asked about, assuming we know which version or artist is being referred to.
Facets: []
Response: Tears for Fears
--------------------------------------------------
Q: How many house of cards seasons are there?
Action: Answer
Reasoning: The question is not ambiguous because it specifies the show "House of Cards" and asks about its seasons. However, without a time frame or version of the show (e.g., which season range), the answer is still uniquely determined as of the most recent complete season count, which is publicly known.
Facets: [Time Frame, Version
--------------------------------------------------
Q: When did the movie come out?
Action: Clarify
Reasoning: The question is ambiguous because it does not specify which movie is being referred to or in which

In [ ]:
# Check your current rejected format
sample = full_df[full_df["action"] == "Clarify"].iloc[0]
print("Current approach:")
print(f"Question: {sample['question']}")
print(f"\nCHOSEN (Clarify):")
print(f"  Reasoning: {sample['reasoning'][:150]}...")
print(f"  Response: {sample['positive_response'][:100]}...")
print(f"\nREJECTED (Answer - but same reasoning?):")
print(f"  Response: {sample['neg_response'][:100]}...")

Current approach:
Question: When did the simpsons first air on television?

CHOSEN (Clarify):
  Reasoning: The question is ambiguous because it does not specify whether 'The Simpsons' refers to the TV series or another type of show, and it does not clarify ...
  Response: Are you asking about the original TV series or a movie, and on which platform (e.g., network, stream...

REJECTED (Answer - but same reasoning?):
  Response: The Simpsons first aired on December 17, 1989....


In [ ]:
hf_REDACTED_TOKEN

In [ ]:
# Use Qwen through HuggingFace Inference API (free)

import pandas as pd
import ast
import requests
import time
from tqdm import tqdm

# HuggingFace Inference API (free tier)
API_URL = "https://api-inference.huggingface.co/models/Qwen/Qwen2.5-72B-Instruct"
headers = {"Authorization": "Bearer hf_REDACTED_TOKEN"}  # Get from hf.co/settings/tokens

In [ ]:
df = pd.read_csv("final_dpo_df.csv")
df_sample = df.sample(n=50, random_state=42).reset_index(drop=True)
print(f"Testing on {len(df_sample)} samples")

def clean_facets(f):
    if isinstance(f, list): return f
    try: return ast.literal_eval(f) if isinstance(ast.literal_eval(f), list) else []
    except: return []

def clean_response(r):
    if r is None: return ""
    if isinstance(r, list): return r[0] if r else ""
    if isinstance(r, str) and r.startswith("["):
        try:
            lst = ast.literal_eval(r)
            return lst[0] if isinstance(lst, list) and lst else r
        except: pass
    return str(r).strip()

def get_wrong_reasoning(question, action):
    if action == "Clarify":
        prompt = f"Generate 2 sentences arguing this question is NOT ambiguous: '{question}'. Start with: The question is not ambiguous because"
    else:
        prompt = f"Generate 2 sentences arguing this question IS ambiguous: '{question}'. Start with: The question is ambiguous because"

    try:
        response = requests.post(API_URL, headers=headers, json={"inputs": prompt, "parameters": {"max_new_tokens": 100}})
        return response.json()[0]["generated_text"].replace(prompt, "").strip()
    except Exception as e:
        print(f"Error: {e}")
        return None


Testing on 50 samples


In [ ]:

improved_data = []
for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    facets = clean_facets(row["facets"])
    facets_str = "[" + ", ".join(facets) + "]" if facets else "[]"
    pos_response = clean_response(row["positive_response"])
    neg_response = clean_response(row["neg_response"])

    chosen = f"Action: {row['action']}\nReasoning: {row['reasoning']}\nFacets: {facets_str}\nResponse: {pos_response}"

    wrong_reasoning = get_wrong_reasoning(row["question"], row["action"])
    if wrong_reasoning is None:
        wrong_reasoning = "The question is clear." if row["action"] == "Clarify" else "The question needs clarification."

    if row["action"] == "Clarify":
        rejected = f"Action: Answer\nReasoning: {wrong_reasoning}\nFacets: []\nResponse: {neg_response}"
    else:
        rejected = f"Action: Clarify\nReasoning: {wrong_reasoning}\nFacets: [Context, Time]\nResponse: Could you provide more details?"

    improved_data.append({"question": row["question"], "chosen": chosen, "rejected": rejected, "gold_action": row["action"]})
    time.sleep(1)  # Light rate limiting

improved_df = pd.DataFrame(improved_data)
improved_df.to_csv("improved_dpo_test_50.csv", index=False)
print(f"\n✅ Saved! Sample:")
print(f"\nCHOSEN:\n{improved_df.iloc[0]['chosen']}")
print(f"\nREJECTED:\n{improved_df.iloc[0]['rejected']}")

  0%|          | 0/50 [00:00<?, ?it/s]

Error: 0


  2%|▏         | 1/50 [00:01<00:52,  1.07s/it]

Error: 0


  4%|▍         | 2/50 [00:02<00:51,  1.06s/it]

...

Error: 0


 98%|█████████▊| 49/50 [00:52<00:01,  1.06s/it]

Error: 0


100%|██████████| 50/50 [00:53<00:00,  1.07s/it]


✅ Saved! Sample:

CHOSEN:
Action: Answer
Reasoning: The question is not ambiguous because it specifies the exact anatomical structure (secondary ossification center in a long bone) being asked about. This description provides enough detail to uniquely determine the location.
Facets: []
Response: epiphyses

REJECTED:
Action: Clarify
Reasoning: The question needs clarification.
Facets: [Context, Time]
Response: Could you provide more details?
